# ⚛️ Phase 2: Physics-Informed Fine-Tuning (PINN)**IMPORTANT: Before running this!**1. Click **"Add Data"** in the top right of Kaggle.2. Click **"Upload Data"** (the up arrow icon).3. Upload your `pretrain_best.pth` file.4. Once uploaded, copy the file path and paste it into the **Phase 2 Training Cell** below (replace `YOUR_UPLOADED_PATH_HERE`).

In [ ]:
!pip install -q "monai[nibabel,tqdm,einops]" nibabel SimpleITK nilearn medpy tensorboard scipy scikit-learnimport torch, osfor d in ["models", "losses", "data", "utils", "checkpoints", "logs", "final_output"]:    os.makedirs(d, exist_ok=True)for d in ["models", "losses", "data", "utils"]:    with open(os.path.join(d, "__init__.py"), "w") as f: f.write("")

In [ ]:
with open("utils/spatial_ops.py", "w") as f:    f.write("""\"\"\"Spatial differential operators for 3D volumes using finite differences.Computes ∇u (gradient), ∇²u (Laplacian), and ∇·(D∇u) (anisotropic diffusion)on regular 3D grids using convolution-based finite difference stencils.All operations are differentiable for backpropagation.\"\"\"import torchimport torch.nn as nnimport torch.nn.functional as Fclass SpatialGradients3D(nn.Module):    \"\"\"    Compute spatial derivatives on 3D volumes using finite differences    implemented as fixed-weight 3D convolutions. This is efficient and    fully differentiable.        Supports:        - First-order central differences: ∂u/∂x, ∂u/∂y, ∂u/∂z        - Second-order central differences: ∂²u/∂x², ∂²u/∂y², ∂²u/∂z²        - 3D Laplacian: ∇²u = ∂²u/∂x² + ∂²u/∂y² + ∂²u/∂z²        - Divergence of flux: ∇·(D(x)∇u)    \"\"\"    def __init__(self, dx: float = 1.0, dy: float = 1.0, dz: float = 1.0):        \"\"\"        Args:            dx, dy, dz: Voxel spacing in mm along each axis.        \"\"\"        super().__init__()        self.dx = dx        self.dy = dy        self.dz = dz        # --- First-order central difference kernels ---        # ∂u/∂x: [-1, 0, 1] / (2*dx) along depth axis        kernel_dx = torch.zeros(1, 1, 3, 3, 3)        kernel_dx[0, 0, 0, 1, 1] = -1.0 / (2.0 * dx)        kernel_dx[0, 0, 2, 1, 1] = 1.0 / (2.0 * dx)        self.register_buffer('kernel_dx', kernel_dx)        # ∂u/∂y: [-1, 0, 1] / (2*dy) along height axis        kernel_dy = torch.zeros(1, 1, 3, 3, 3)        kernel_dy[0, 0, 1, 0, 1] = -1.0 / (2.0 * dy)        kernel_dy[0, 0, 1, 2, 1] = 1.0 / (2.0 * dy)        self.register_buffer('kernel_dy', kernel_dy)        # ∂u/∂z: [-1, 0, 1] / (2*dz) along width axis        kernel_dz = torch.zeros(1, 1, 3, 3, 3)        kernel_dz[0, 0, 1, 1, 0] = -1.0 / (2.0 * dz)        kernel_dz[0, 0, 1, 1, 2] = 1.0 / (2.0 * dz)        self.register_buffer('kernel_dz', kernel_dz)        # --- Second-order central difference kernels ---        # ∂²u/∂x²: [1, -2, 1] / dx² along depth axis        kernel_dxx = torch.zeros(1, 1, 3, 3, 3)        kernel_dxx[0, 0, 0, 1, 1] = 1.0 / (dx * dx)        kernel_dxx[0, 0, 1, 1, 1] = -2.0 / (dx * dx)        kernel_dxx[0, 0, 2, 1, 1] = 1.0 / (dx * dx)        self.register_buffer('kernel_dxx', kernel_dxx)        # ∂²u/∂y²: [1, -2, 1] / dy² along height axis        kernel_dyy = torch.zeros(1, 1, 3, 3, 3)        kernel_dyy[0, 0, 1, 0, 1] = 1.0 / (dy * dy)        kernel_dyy[0, 0, 1, 1, 1] = -2.0 / (dy * dy)        kernel_dyy[0, 0, 1, 2, 1] = 1.0 / (dy * dy)        self.register_buffer('kernel_dyy', kernel_dyy)        # ∂²u/∂z²: [1, -2, 1] / dz² along width axis        kernel_dzz = torch.zeros(1, 1, 3, 3, 3)        kernel_dzz[0, 0, 1, 1, 0] = 1.0 / (dz * dz)        kernel_dzz[0, 0, 1, 1, 1] = -2.0 / (dz * dz)        kernel_dzz[0, 0, 1, 1, 2] = 1.0 / (dz * dz)        self.register_buffer('kernel_dzz', kernel_dzz)        # --- 7-point 3D Laplacian stencil (isotropic case) ---        kernel_lap = torch.zeros(1, 1, 3, 3, 3)        kernel_lap[0, 0, 1, 1, 1] = -(2.0/(dx*dx) + 2.0/(dy*dy) + 2.0/(dz*dz))        kernel_lap[0, 0, 0, 1, 1] = 1.0 / (dx * dx)        kernel_lap[0, 0, 2, 1, 1] = 1.0 / (dx * dx)        kernel_lap[0, 0, 1, 0, 1] = 1.0 / (dy * dy)        kernel_lap[0, 0, 1, 2, 1] = 1.0 / (dy * dy)        kernel_lap[0, 0, 1, 1, 0] = 1.0 / (dz * dz)        kernel_lap[0, 0, 1, 1, 2] = 1.0 / (dz * dz)        self.register_buffer('kernel_laplacian', kernel_lap)    def gradient(self, u: torch.Tensor) -> tuple:        \"\"\"        Compute first-order spatial gradient ∇u = (∂u/∂x, ∂u/∂y, ∂u/∂z).        Args:            u: Tensor of shape [B, 1, D, H, W] — tumor density field.        Returns:            (du_dx, du_dy, du_dz): Each [B, 1, D, H, W].        \"\"\"        du_dx = F.conv3d(u, self.kernel_dx, padding=1)        du_dy = F.conv3d(u, self.kernel_dy, padding=1)        du_dz = F.conv3d(u, self.kernel_dz, padding=1)        return du_dx, du_dy, du_dz    def laplacian(self, u: torch.Tensor) -> torch.Tensor:        \"\"\"        Compute the 3D Laplacian ∇²u using the 7-point stencil.        Args:            u: Tensor of shape [B, 1, D, H, W].        Returns:            Tensor of shape [B, 1, D, H, W] — the Laplacian.        \"\"\"        return F.conv3d(u, self.kernel_laplacian, padding=1)    def second_derivatives(self, u: torch.Tensor) -> tuple:        \"\"\"        Compute second-order partial derivatives.        Returns:            (d²u/dx², d²u/dy², d²u/dz²): Each [B, 1, D, H, W].        \"\"\"        uxx = F.conv3d(u, self.kernel_dxx, padding=1)        uyy = F.conv3d(u, self.kernel_dyy, padding=1)        uzz = F.conv3d(u, self.kernel_dzz, padding=1)        return uxx, uyy, uzz    def divergence_of_flux(self, u: torch.Tensor, D: torch.Tensor) -> torch.Tensor:        \"\"\"        Compute ∇·(D(x)∇u) — divergence of the diffusion flux.                Uses the product rule: ∇·(D∇u) = D·∇²u + ∇D·∇u        Args:            u: Tumor density [B, 1, D, H, W].            D: Spatially-varying diffusion coefficient [B, 1, D, H, W].        Returns:            Tensor [B, 1, D, H, W] — the divergence of the flux.        \"\"\"        # Compute ∇u        du_dx, du_dy, du_dz = self.gradient(u)        # Compute ∇D        dD_dx, dD_dy, dD_dz = self.gradient(D)        # Compute ∇²u        lap_u = self.laplacian(u)        # ∇·(D∇u) = D·∇²u + ∇D·∇u        divergence = D * lap_u + (dD_dx * du_dx + dD_dy * du_dy + dD_dz * du_dz)        return divergence    def gradient_magnitude(self, u: torch.Tensor) -> torch.Tensor:        \"\"\"        Compute |∇u| = sqrt((∂u/∂x)² + (∂u/∂y)² + (∂u/∂z)²).        Args:            u: Tensor [B, 1, D, H, W].        Returns:            Tensor [B, 1, D, H, W] — gradient magnitude.        \"\"\"        du_dx, du_dy, du_dz = self.gradient(u)        return torch.sqrt(du_dx**2 + du_dy**2 + du_dz**2 + 1e-8)    def normal_gradient_at_boundary(self, u: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:        \"\"\"        Compute ∇u·n at the boundary of the domain.                For a regular grid, the boundary is the edge voxels.        The outward normal at edges is along the axis direction.        Args:            u: Tensor [B, 1, D, H, W].            mask: Optional brain mask [B, 1, D, H, W]. If given,                  boundary = edge of the mask.        Returns:            Tensor of boundary normal gradients [B, 1, D, H, W].            Non-zero only at boundary voxels.        \"\"\"        du_dx, du_dy, du_dz = self.gradient(u)        if mask is not None:            # Boundary = voxels in mask that neighbor voxels outside mask            # Erode mask by 1 voxel            erode_kernel = torch.ones(1, 1, 3, 3, 3, device=u.device) / 27.0            eroded = F.conv3d(mask.float(), erode_kernel, padding=1)            boundary = mask.float() * (1.0 - (eroded > 0.99).float())        else:            # Use volume edges as boundary            boundary = torch.zeros_like(u)            boundary[:, :, 0, :, :] = 1.0   # front face            boundary[:, :, -1, :, :] = 1.0  # back face            boundary[:, :, :, 0, :] = 1.0   # top face            boundary[:, :, :, -1, :] = 1.0  # bottom face            boundary[:, :, :, :, 0] = 1.0   # left face            boundary[:, :, :, :, -1] = 1.0  # right face        # At boundaries, approximate ∇u·n using the gradient magnitude        grad_mag = torch.sqrt(du_dx**2 + du_dy**2 + du_dz**2 + 1e-8)        return grad_mag * boundarydef compute_temporal_derivative(u_t1: torch.Tensor, u_t2: torch.Tensor, delta_t: float) -> torch.Tensor:    \"\"\"    Approximate ∂u/∂t using forward difference between two time points.    Args:        u_t1: Tumor density at time t₁ [B, 1, D, H, W].        u_t2: Tumor density at time t₂ [B, 1, D, H, W].        delta_t: Time difference (t₂ - t₁) in days.    Returns:        Tensor [B, 1, D, H, W] — approximate ∂u/∂t.    \"\"\"    return (u_t2 - u_t1) / (delta_t + 1e-8)""")

In [ ]:
with open("utils/metrics.py", "w") as f:    f.write("""\"\"\"Evaluation metrics for brain tumor analysis.Includes:    - Dice Score (per-class and mean)    - Hausdorff Distance (95th percentile)    - Growth Prediction Error (MSE, MAE on tumor density)    - Physics Residual Error (PDE residual norm)    - Volume-based metrics\"\"\"import numpy as npimport torchfrom scipy import ndimagedef compute_dice(pred: torch.Tensor, target: torch.Tensor,                  num_classes: int = 4, include_background: bool = False,                 smooth: float = 1e-5) -> dict:    \"\"\"    Compute Dice similarity coefficient per class.    Args:        pred: Predicted segmentation [B, num_classes, D, H, W] (probabilities or logits).        target: Ground truth segmentation [B, 1, D, H, W] (integer labels) or one-hot.        num_classes: Number of segmentation classes.        include_background: Whether to include background in mean Dice.        smooth: Smoothing constant to avoid division by zero.    Returns:        Dictionary with per-class Dice and mean Dice.    \"\"\"    if pred.dim() == 5 and pred.shape[1] > 1:        pred = torch.argmax(pred, dim=1)  # [B, D, H, W]    elif pred.dim() == 5:        pred = pred.squeeze(1)    if target.dim() == 5:        target = target.squeeze(1)    dice_scores = {}    start_class = 0 if include_background else 1    for c in range(start_class, num_classes):        pred_c = (pred == c).float()        target_c = (target == c).float()        intersection = (pred_c * target_c).sum()        union = pred_c.sum() + target_c.sum()        dice = (2.0 * intersection + smooth) / (union + smooth)        dice_scores[f"dice_class_{c}"] = dice.item()    # BraTS-specific regions    if num_classes == 4:        # Whole Tumor (WT): classes 1, 2, 3 (NCR/NET + ED + ET)        pred_wt = ((pred == 1) | (pred == 2) | (pred == 3)).float()        target_wt = ((target == 1) | (target == 2) | (target == 3)).float()        dice_wt = (2.0 * (pred_wt * target_wt).sum() + smooth) / (pred_wt.sum() + target_wt.sum() + smooth)        dice_scores["dice_WT"] = dice_wt.item()        # Tumor Core (TC): classes 1, 3 (NCR/NET + ET)        pred_tc = ((pred == 1) | (pred == 3)).float()        target_tc = ((target == 1) | (target == 3)).float()        dice_tc = (2.0 * (pred_tc * target_tc).sum() + smooth) / (pred_tc.sum() + target_tc.sum() + smooth)        dice_scores["dice_TC"] = dice_tc.item()        # Enhancing Tumor (ET): class 3        pred_et = (pred == 3).float()        target_et = (target == 3).float()        dice_et = (2.0 * (pred_et * target_et).sum() + smooth) / (pred_et.sum() + target_et.sum() + smooth)        dice_scores["dice_ET"] = dice_et.item()        dice_scores["dice_mean"] = np.mean([dice_wt.item(), dice_tc.item(), dice_et.item()])    else:        vals = [v for k, v in dice_scores.items() if k.startswith("dice_class")]        dice_scores["dice_mean"] = np.mean(vals) if vals else 0.0    return dice_scoresdef compute_hausdorff(pred: np.ndarray, target: np.ndarray,                       percentile: float = 95.0,                      voxel_spacing: tuple = (1.0, 1.0, 1.0)) -> float:    \"\"\"    Compute the Hausdorff distance (at a given percentile) between    predicted and target binary masks.    Args:        pred: Binary prediction mask [D, H, W].        target: Binary ground truth mask [D, H, W].        percentile: Percentile for robust Hausdorff (default 95).        voxel_spacing: Physical voxel spacing in mm.    Returns:        Hausdorff distance in mm.    \"\"\"    if np.sum(pred) == 0 and np.sum(target) == 0:        return 0.0    if np.sum(pred) == 0 or np.sum(target) == 0:        return np.inf    # Compute distance transforms    pred_border = _get_surface_points(pred)    target_border = _get_surface_points(target)    # Get surface point coordinates    pred_coords = np.argwhere(pred_border) * np.array(voxel_spacing)    target_coords = np.argwhere(target_border) * np.array(voxel_spacing)    if len(pred_coords) == 0 or len(target_coords) == 0:        return np.inf    # Compute distances from pred surface to target surface and vice versa    from scipy.spatial.distance import cdist    d_pred_to_target = cdist(pred_coords, target_coords).min(axis=1)    d_target_to_pred = cdist(target_coords, pred_coords).min(axis=1)    all_distances = np.concatenate([d_pred_to_target, d_target_to_pred])    return np.percentile(all_distances, percentile)def _get_surface_points(mask: np.ndarray) -> np.ndarray:    \"\"\"Extract surface voxels of a binary mask using erosion.\"\"\"    struct = ndimage.generate_binary_structure(3, 1)    eroded = ndimage.binary_erosion(mask, structure=struct)    return mask.astype(bool) ^ eroded.astype(bool)def compute_growth_prediction_error(pred_density: torch.Tensor,                                      target_density: torch.Tensor,                                     mask: torch.Tensor = None) -> dict:    \"\"\"    Compute tumor growth prediction errors.    Args:        pred_density: Predicted tumor density u(x,t₂) [B, 1, D, H, W].        target_density: Observed tumor density at t₂ [B, 1, D, H, W].        mask: Optional brain mask [B, 1, D, H, W].    Returns:        Dictionary with MSE, MAE, volume error, and spatial correlation.    \"\"\"    if mask is not None:        pred_masked = pred_density * mask        target_masked = target_density * mask        n_voxels = mask.sum().clamp(min=1)    else:        pred_masked = pred_density        target_masked = target_density        n_voxels = pred_density.numel()    # MSE    mse = ((pred_masked - target_masked) ** 2).sum() / n_voxels    # MAE    mae = (pred_masked - target_masked).abs().sum() / n_voxels    # Volume error (in voxels)    pred_volume = (pred_masked > 0.5).float().sum()    target_volume = (target_masked > 0.5).float().sum()    volume_error = (pred_volume - target_volume).abs()    volume_relative_error = volume_error / target_volume.clamp(min=1)    # Spatial Pearson correlation    pred_flat = pred_masked.flatten()    target_flat = target_masked.flatten()    pred_centered = pred_flat - pred_flat.mean()    target_centered = target_flat - target_flat.mean()    correlation = (pred_centered * target_centered).sum() / (        pred_centered.norm() * target_centered.norm() + 1e-8    )    return {        "growth_mse": mse.item(),        "growth_mae": mae.item(),        "volume_error_voxels": volume_error.item(),        "volume_relative_error": volume_relative_error.item(),        "spatial_correlation": correlation.item(),    }def compute_physics_residual(u: torch.Tensor, D: torch.Tensor, rho: torch.Tensor,                              spatial_ops, du_dt: torch.Tensor = None,                              pde_model: str = "fisher_kpp",                              mask: torch.Tensor = None) -> dict:    \"\"\"    Compute how well the predicted tumor density satisfies the PDE.    Args:        u: Predicted tumor density [B, 1, D, H, W].        D: Diffusion coefficient [B, 1, D, H, W].        rho: Proliferation rate [B, 1, D, H, W] or scalar.        spatial_ops: SpatialGradients3D instance.        du_dt: Temporal derivative [B, 1, D, H, W], or None (assumes steady state).        pde_model: "fisher_kpp" or "gompertz".        mask: Optional brain mask.    Returns:        Dictionary with mean and max PDE residual.    \"\"\"    # Compute diffusion term: ∇·(D∇u)    diffusion_term = spatial_ops.divergence_of_flux(u, D)    # Compute reaction term    if pde_model == "fisher_kpp":        reaction_term = rho * u * (1.0 - u)    elif pde_model == "gompertz":        eps = 1e-7        reaction_term = rho * u * torch.log((1.0) / (u + eps))    else:        raise ValueError(f"Unknown PDE model: {pde_model}")    # PDE: ∂u/∂t = ∇·(D∇u) + reaction    rhs = diffusion_term + reaction_term    if du_dt is not None:        residual = du_dt - rhs    else:        # Steady-state assumption: ∂u/∂t ≈ 0        residual = -rhs    if mask is not None:        residual = residual * mask        n = mask.sum().clamp(min=1)    else:        n = residual.numel()    mean_residual = (residual ** 2).sum() / n    max_residual = residual.abs().max()    return {        "pde_residual_mean": mean_residual.item(),        "pde_residual_max": max_residual.item(),        "pde_residual_l1": (residual.abs().sum() / n).item(),    }def compute_all_metrics(pred_seg, target_seg, pred_density, target_density,                         u, D, rho, spatial_ops, du_dt=None,                         pde_model="fisher_kpp", mask=None,                         num_classes=4) -> dict:    \"\"\"Compute all evaluation metrics in one call.\"\"\"    metrics = {}    # Segmentation metrics    if pred_seg is not None and target_seg is not None:        dice = compute_dice(pred_seg, target_seg, num_classes=num_classes)        metrics.update(dice)    # Growth prediction metrics    if pred_density is not None and target_density is not None:        growth = compute_growth_prediction_error(pred_density, target_density, mask)        metrics.update(growth)    # Physics residual    if u is not None and D is not None and rho is not None:        physics = compute_physics_residual(u, D, rho, spatial_ops, du_dt, pde_model, mask)        metrics.update(physics)    return metrics""")

In [ ]:
with open("utils/ema.py", "w") as f:    f.write("""\"\"\"Exponential Moving Average (EMA) for model weights.Improves model generalization and stability during training.\"\"\"import copyimport torchimport torch.nn as nnclass EMAModel(nn.Module):    \"\"\"    Maintains an Exponential Moving Average of a model's weights.    \"\"\"    def __init__(self, model: nn.Module, decay: float = 0.999, device=None):        super().__init__()        self.decay = decay        # Create a deep copy of the model for EMA        self.ema_model = copy.deepcopy(model)        self.ema_model.eval()                if device is not None:            self.ema_model = self.ema_model.to(device)        # Disable gradients for EMA model        for param in self.ema_model.parameters():            param.requires_grad = False    @torch.no_grad()    def update(self, model: nn.Module):        \"\"\"        Update the EMA weights based on the current model weights.        \"\"\"        for ema_param, param in zip(self.ema_model.parameters(), model.parameters()):            if param.requires_grad:                ema_param.data.mul_(self.decay).add_(param.data, alpha=1.0 - self.decay)    def forward(self, *args, **kwargs):        \"\"\"        Forward pass using EMA weights.        \"\"\"        return self.ema_model(*args, **kwargs)    def state_dict(self, *args, **kwargs):        \"\"\"        Return the state dict of the EMA model.        \"\"\"        return self.ema_model.state_dict(*args, **kwargs)    def load_state_dict(self, state_dict, strict=True):        \"\"\"        Load a state dict into the EMA model.        \"\"\"        return self.ema_model.load_state_dict(state_dict, strict=strict)""")

In [ ]:
with open("models/attention.py", "w") as f:    f.write("""\"\"\"Attention mechanisms for 3D medical image analysis.    - Squeeze-and-Excitation (SE) channel attention    - Spatial Attention Gate (for skip connections)    - CBAM (Convolutional Block Attention Module) adapted for 3D\"\"\"import torchimport torch.nn as nnimport torch.nn.functional as Fclass SqueezeExcitation3D(nn.Module):    \"\"\"Channel attention via global pooling → FC → sigmoid reweighting.\"\"\"    def __init__(self, channels: int, reduction: int = 16):        super().__init__()        mid = max(channels // reduction, 4)        self.squeeze = nn.AdaptiveAvgPool3d(1)        self.excite = nn.Sequential(            nn.Linear(channels, mid, bias=False),            nn.ReLU(inplace=True),            nn.Linear(mid, channels, bias=False),            nn.Sigmoid(),        )    def forward(self, x):        b, c = x.shape[:2]        w = self.squeeze(x).view(b, c)        w = self.excite(w).view(b, c, 1, 1, 1)        return x * wclass AttentionGate3D(nn.Module):    \"\"\"    Attention gate for skip connections (Oktay et al., 2018).    Learns to suppress irrelevant regions in skip features using    the gating signal from the decoder. Critical for focusing on    tumor regions and ignoring healthy tissue.        α = σ(W_ψ(ReLU(W_g·g + W_x·x + b)) + b_ψ)        output = x * α    \"\"\"    def __init__(self, gate_ch: int, skip_ch: int, inter_ch: int = None):        super().__init__()        inter_ch = inter_ch or skip_ch // 2        self.W_gate = nn.Conv3d(gate_ch, inter_ch, 1, bias=False)        self.W_skip = nn.Conv3d(skip_ch, inter_ch, 1, bias=False)        self.psi = nn.Sequential(            nn.Conv3d(inter_ch, 1, 1, bias=False),            nn.Sigmoid(),        )        self.relu = nn.ReLU(inplace=True)        self.norm = nn.InstanceNorm3d(inter_ch)    def forward(self, gate: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:        g = self.W_gate(gate)        # Upsample gate to match skip spatial size        if g.shape[2:] != skip.shape[2:]:            g = F.interpolate(g, size=skip.shape[2:], mode="trilinear", align_corners=False)        x = self.W_skip(skip)        attn = self.relu(g + x)        attn = self.norm(attn)        attn = self.psi(attn)        return skip * attnclass SpatialAttention3D(nn.Module):    \"\"\"Spatial attention using max+avg pooling across channels.\"\"\"    def __init__(self, kernel_size: int = 7):        super().__init__()        pad = kernel_size // 2        self.conv = nn.Conv3d(2, 1, kernel_size, padding=pad, bias=False)        self.sigmoid = nn.Sigmoid()    def forward(self, x):        avg_pool = x.mean(dim=1, keepdim=True)        max_pool = x.max(dim=1, keepdim=True)[0]        combined = torch.cat([avg_pool, max_pool], dim=1)        attn = self.sigmoid(self.conv(combined))        return x * attnclass CBAM3D(nn.Module):    \"\"\"Convolutional Block Attention Module: channel + spatial attention.\"\"\"    def __init__(self, channels: int, reduction: int = 16):        super().__init__()        self.channel_attn = SqueezeExcitation3D(channels, reduction)        self.spatial_attn = SpatialAttention3D(kernel_size=7)    def forward(self, x):        x = self.channel_attn(x)        x = self.spatial_attn(x)        return x""")

In [ ]:
with open("models/resnet_backbone.py", "w") as f:    f.write("""\"\"\"MONAI 3D ResNet Encoder with multi-scale feature extraction.Wraps MONAI's ResNet to expose intermediate features at multiplespatial resolutions for the decoder (skip connections).Architecture (ResNet50):    Input: [B, 4, 128, 128, 128]        stem  (conv1 + bn + relu + pool):  [B, 64,  64, 64, 64]    layer1 (3 bottleneck blocks):      [B, 256, 32, 32, 32]    layer2 (4 bottleneck blocks):      [B, 512, 16, 16, 16]    layer3 (6 bottleneck blocks):      [B, 1024, 8,  8,  8]    layer4 (3 bottleneck blocks):      [B, 2048, 4,  4,  4]\"\"\"import torchimport torch.nn as nnfrom collections import OrderedDictfrom monai.networks.nets import ResNet, resnet10, resnet18, resnet34, resnet50, resnet101, resnet152, resnet200# Channel dimensions for each ResNet variant at each stageRESNET_CHANNELS = {    "resnet10":  {"stem": 64, "layer1": 64,  "layer2": 128, "layer3": 256, "layer4": 512},    "resnet18":  {"stem": 64, "layer1": 64,  "layer2": 128, "layer3": 256, "layer4": 512},    "resnet34":  {"stem": 64, "layer1": 64,  "layer2": 128, "layer3": 256, "layer4": 512},    "resnet50":  {"stem": 64, "layer1": 256, "layer2": 512, "layer3": 1024, "layer4": 2048},    "resnet101": {"stem": 64, "layer1": 256, "layer2": 512, "layer3": 1024, "layer4": 2048},    "resnet152": {"stem": 64, "layer1": 256, "layer2": 512, "layer3": 1024, "layer4": 2048},    "resnet200": {"stem": 64, "layer1": 256, "layer2": 512, "layer3": 1024, "layer4": 2048},}# Block type for each variantRESNET_BLOCKS = {    "resnet10": "basic", "resnet18": "basic", "resnet34": "basic",    "resnet50": "bottleneck", "resnet101": "bottleneck", "resnet152": "bottleneck",    "resnet200": "bottleneck",}# Layer configsRESNET_LAYERS = {    "resnet10": [1, 1, 1, 1], "resnet18": [2, 2, 2, 2], "resnet34": [3, 4, 6, 3],    "resnet50": [3, 4, 6, 3], "resnet101": [3, 4, 23, 3], "resnet152": [3, 8, 36, 3],    "resnet200": [3, 24, 36, 3],}class ResNetEncoder(nn.Module):    \"\"\"    3D ResNet encoder that returns multi-scale feature maps.        Wraps MONAI's ResNet and provides an interface to extract    features at each resolution level for skip connections.    \"\"\"    def __init__(self, variant: str = "resnet50", in_channels: int = 4,                 pretrained_path: str = None, freeze_layers: int = 0):        \"\"\"        Args:            variant: ResNet variant name (e.g., "resnet50").            in_channels: Number of input channels (4 for multi-modal MRI).            pretrained_path: Path to pretrained weights (.pth file).            freeze_layers: Number of initial layers to freeze (0-5).                0 = train all, 1 = freeze stem, 2 = freeze stem+layer1, etc.        \"\"\"        super().__init__()        self.variant = variant        self.channels = RESNET_CHANNELS[variant]        # Build the ResNet backbone        block_type = RESNET_BLOCKS[variant]        layers = RESNET_LAYERS[variant]        if block_type == "bottleneck":            block_inplanes = [64, 128, 256, 512]        else:            block_inplanes = [64, 128, 256, 512]        self.backbone = ResNet(            block=block_type,            layers=layers,            block_inplanes=block_inplanes,            spatial_dims=3,            n_input_channels=in_channels,            num_classes=1,  # Placeholder — we won't use the FC layer            conv1_t_stride=2,        )        # Load pretrained weights if provided        if pretrained_path:            self._load_pretrained(pretrained_path, in_channels)        # Freeze layers        if freeze_layers > 0:            self._freeze_layers(freeze_layers)    def _load_pretrained(self, weights_path: str, in_channels: int):        \"\"\"Load pretrained weights with flexible key matching.\"\"\"        print(f"Loading pretrained weights from {weights_path}...")        state_dict = torch.load(weights_path, map_location="cpu")        # Handle DataParallel prefix        if any(k.startswith("module.") for k in state_dict.keys()):            state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}        model_dict = self.backbone.state_dict()        # Filter compatible weights        compatible = {}        for k, v in state_dict.items():            if k in model_dict:                if v.shape == model_dict[k].shape:                    compatible[k] = v                elif k == "conv1.weight" and v.shape[1] != in_channels:                    # Handle channel mismatch in first conv                    print(f"  Adapting conv1: {v.shape[1]}ch → {in_channels}ch")                    if in_channels > v.shape[1]:                        # Repeat channels                        repeats = in_channels // v.shape[1] + 1                        v = v.repeat(1, repeats, 1, 1, 1)[:, :in_channels]                    else:                        v = v[:, :in_channels]                    compatible[k] = v                else:                    print(f"  Skipping {k}: shape mismatch {v.shape} vs {model_dict[k].shape}")        model_dict.update(compatible)        self.backbone.load_state_dict(model_dict, strict=False)        print(f"  Loaded {len(compatible)}/{len(model_dict)} parameters")    def _freeze_layers(self, n_layers: int):        \"\"\"Freeze the first n_layers of the network.\"\"\"        layer_groups = [            [self.backbone.conv1, self.backbone.bn1],            [self.backbone.layer1],            [self.backbone.layer2],            [self.backbone.layer3],            [self.backbone.layer4],        ]        for i in range(min(n_layers, len(layer_groups))):            for module in layer_groups[i]:                for param in module.parameters():                    param.requires_grad = False            print(f"  Froze layer group {i}")    def forward(self, x: torch.Tensor) -> dict:        \"\"\"        Extract multi-scale features from the input volume.        Args:            x: Input tensor [B, C, D, H, W] where C = num modalities.        Returns:            Dictionary with feature maps at each scale:                "stem":   [B, 64,   D/2,  H/2,  W/2]                "layer1": [B, 256,  D/4,  H/4,  W/4]   (ResNet50)                "layer2": [B, 512,  D/8,  H/8,  W/8]                "layer3": [B, 1024, D/16, H/16, W/16]                "layer4": [B, 2048, D/32, H/32, W/32]        \"\"\"        features = {}        # Stem        x = self.backbone.conv1(x)        x = self.backbone.bn1(x)        x = self.backbone.act(x)        features["stem"] = x  # [B, 64, D/2, H/2, W/2]        x = self.backbone.maxpool(x)        # Residual layers        x = self.backbone.layer1(x)        features["layer1"] = x  # [B, 256, D/4, H/4, W/4]        x = self.backbone.layer2(x)        features["layer2"] = x  # [B, 512, D/8, H/8, W/8]        x = self.backbone.layer3(x)        features["layer3"] = x  # [B, 1024, D/16, H/16, W/16]        x = self.backbone.layer4(x)        features["layer4"] = x  # [B, 2048, D/32, H/32, W/32]        return features    def get_channels(self) -> dict:        \"\"\"Return the channel dimensions at each feature level.\"\"\"        return self.channels.copy()""")

In [ ]:
with open("models/decoder.py", "w") as f:    f.write("""\"\"\"Decoder modules for dense prediction from ResNet encoder features.Includes:    - TumorDensityDecoder: Predicts u(x,t) field [B, 1, D, H, W]    - PhysicsParameterHead: Predicts D(x), ρ(x) spatially    - SegmentationHead: Predicts segmentation classes (pretraining)    Uses transposed convolutions + skip connections for upsampling.\"\"\"import torchimport torch.nn as nnimport torch.nn.functional as Ffrom .attention import AttentionGate3D, CBAM3Dclass ConvBlock3D(nn.Module):    \"\"\"3D convolution + InstanceNorm + LeakyReLU block.\"\"\"    def __init__(self, in_ch: int, out_ch: int, kernel_size: int = 3,                  padding: int = 1, dropout: float = 0.0):        super().__init__()        self.conv = nn.Sequential(            nn.Conv3d(in_ch, out_ch, kernel_size, padding=padding, bias=False),            nn.InstanceNorm3d(out_ch, affine=True),            nn.LeakyReLU(0.01, inplace=True),            nn.Dropout3d(dropout) if dropout > 0 else nn.Identity(),            nn.Conv3d(out_ch, out_ch, kernel_size, padding=padding, bias=False),            nn.InstanceNorm3d(out_ch, affine=True),        )        self.residual = nn.Sequential()        if in_ch != out_ch:            self.residual = nn.Sequential(                nn.Conv3d(in_ch, out_ch, kernel_size=1, bias=False),                nn.InstanceNorm3d(out_ch, affine=True)            )        self.relu = nn.LeakyReLU(0.01, inplace=True)    def forward(self, x):        return self.relu(self.conv(x) + self.residual(x))class UpBlock3D(nn.Module):    \"\"\"Upsample + concatenate skip connection + ConvBlock.\"\"\"    def __init__(self, in_ch: int, skip_ch: int, out_ch: int, dropout: float = 0.0, use_attention: bool = True):        \"\"\"        Args:            in_ch: Input channels from previous decoder level.            skip_ch: Channels from encoder skip connection.            out_ch: Output channels.            use_attention: Whether to use Spatial Attention Gate on skip connection.        \"\"\"        super().__init__()        self.upsample = nn.ConvTranspose3d(in_ch, in_ch, kernel_size=2, stride=2)                self.use_attention = use_attention        if use_attention and skip_ch > 0:            self.attn_gate = AttentionGate3D(gate_ch=in_ch, skip_ch=skip_ch)        else:            self.attn_gate = None                    self.conv = ConvBlock3D(in_ch + skip_ch, out_ch, dropout=dropout)    def forward(self, x, skip=None):        x = self.upsample(x)                if skip is not None:            # Handle size mismatch due to odd dimensions            if x.shape != skip.shape:                diff_d = skip.shape[2] - x.shape[2]                diff_h = skip.shape[3] - x.shape[3]                diff_w = skip.shape[4] - x.shape[4]                x = F.pad(x, [                    diff_w // 2, diff_w - diff_w // 2,                    diff_h // 2, diff_h - diff_h // 2,                    diff_d // 2, diff_d - diff_d // 2,                ])                        if self.attn_gate is not None:                skip = self.attn_gate(gate=x, skip=skip)                            x = torch.cat([x, skip], dim=1)                return self.conv(x)class TumorDensityDecoder(nn.Module):    \"\"\"    Decoder that produces a dense tumor density field u(x,t) ∈ [0, 1].        Takes multi-scale features from the ResNet encoder and upsamples    to full resolution with skip connections.        Architecture (for ResNet50 encoder):        layer4 [2048, 4,4,4] → up1 [256, 8,8,8]    + skip from layer3        up1    [256, 8,8,8]  → up2 [128, 16,16,16]  + skip from layer2        up2    [128, 16,16,16]→ up3 [64, 32,32,32]   + skip from layer1        up3    [64, 32,32,32] → up4 [32, 64,64,64]   + skip from stem        up4    [32, 64,64,64] → up5 [16, 128,128,128] (no skip)        up5    → conv1x1 → [1, 128,128,128] → sigmoid    \"\"\"    def __init__(self, encoder_channels: dict, decoder_channels=(256, 128, 64, 32, 16),                 dropout: float = 0.1, deep_supervision: bool = True):        \"\"\"        Args:            encoder_channels: Dict with keys "stem", "layer1", ..., "layer4"                              mapping to channel counts.            decoder_channels: Tuple of channel counts for each decoder level.            dropout: Dropout rate in decoder blocks.        \"\"\"        super().__init__()        enc = encoder_channels        dec = decoder_channels        # Bottleneck reduction        self.bottleneck = ConvBlock3D(enc["layer4"], dec[0], dropout=dropout)        # Decoder levels (from deepest to shallowest)        self.up1 = UpBlock3D(dec[0], enc["layer3"], dec[0], dropout=dropout)        self.up2 = UpBlock3D(dec[0], enc["layer2"], dec[1], dropout=dropout)        self.up3 = UpBlock3D(dec[1], enc["layer1"], dec[2], dropout=dropout)        self.up4 = UpBlock3D(dec[2], enc["stem"],   dec[3], dropout=dropout)        self.up5 = UpBlock3D(dec[3], 0,             dec[4], dropout=0.0)        # Final 1x1 conv → tumor density        self.final_conv = nn.Conv3d(dec[4], 1, kernel_size=1)                # Deep supervision heads        self.deep_supervision = deep_supervision        if self.deep_supervision:            self.ds2 = nn.Conv3d(dec[3], 1, kernel_size=1)            self.ds3 = nn.Conv3d(dec[2], 1, kernel_size=1)    def forward(self, features: dict, return_ds: bool = False):        \"\"\"        Args:            features: Dict of encoder features with keys:                "stem", "layer1", "layer2", "layer3", "layer4"            return_ds: Whether to return deep supervision list.        Returns:            Tumor density field u(x,t) [B, 1, D, H, W] ∈ [0, 1].            Or if return_ds is True, a list of outputs at different resolutions.        \"\"\"        x = self.bottleneck(features["layer4"])        x = self.up1(x, features["layer3"])        x = self.up2(x, features["layer2"])        x = self.up3(x, features["layer1"])        ds3 = x                x = self.up4(x, features["stem"])        ds2 = x                x = self.up5(x)  # No skip connection — full resolution        out = torch.sigmoid(self.final_conv(x))  # Ensure u ∈ [0, 1]                if self.deep_supervision and return_ds:            return [out, torch.sigmoid(self.ds2(ds2)), torch.sigmoid(self.ds3(ds3))]        return outclass PhysicsParameterHead(nn.Module):    \"\"\"    Predicts spatially-varying physics parameters from encoder features.        Outputs:        - D(x): Diffusion coefficient field [B, 1, D, H, W]        - ρ(x): Proliferation rate field [B, 1, D, H, W]        Both outputs are constrained to positive values via softplus activation,    and clamped to physically meaningful ranges.    \"\"\"    def __init__(self, encoder_channels: dict,                 diffusion_range=(0.0001, 0.5),                 proliferation_range=(0.001, 0.5)):        \"\"\"        Args:            encoder_channels: Dict of channel counts from encoder.            diffusion_range: (min, max) for D(x) in mm²/day.            proliferation_range: (min, max) for ρ(x) in 1/day.        \"\"\"        super().__init__()        self.d_min, self.d_max = diffusion_range        self.rho_min, self.rho_max = proliferation_range        # Light decoder from layer3 features (coarser is fine for smooth params)        in_ch = encoder_channels["layer3"]                self.shared = nn.Sequential(            nn.Conv3d(in_ch, 128, 3, padding=1, bias=False),            nn.InstanceNorm3d(128, affine=True),            nn.LeakyReLU(0.01, inplace=True),            nn.Conv3d(128, 64, 3, padding=1, bias=False),            nn.InstanceNorm3d(64, affine=True),            nn.LeakyReLU(0.01, inplace=True),        )        # Diffusion head        self.diffusion_head = nn.Sequential(            nn.Conv3d(64, 32, 3, padding=1),            nn.LeakyReLU(0.01, inplace=True),            nn.Conv3d(32, 1, 1),        )        # Proliferation head        self.proliferation_head = nn.Sequential(            nn.Conv3d(64, 32, 3, padding=1),            nn.LeakyReLU(0.01, inplace=True),            nn.Conv3d(32, 1, 1),        )    def forward(self, features: dict, target_size: tuple = None) -> tuple:        \"\"\"        Args:            features: Encoder features dict.            target_size: Target spatial size for upsampling (D, H, W).        Returns:            (D_field, rho_field): Each [B, 1, D, H, W], positive-valued.        \"\"\"        x = self.shared(features["layer3"])        # Diffusion coefficient        D = self.diffusion_head(x)        D = torch.sigmoid(D) * (self.d_max - self.d_min) + self.d_min        # Proliferation rate        rho = self.proliferation_head(x)        rho = torch.sigmoid(rho) * (self.rho_max - self.rho_min) + self.rho_min        # Upsample to target resolution        if target_size is not None:            D = F.interpolate(D, size=target_size, mode="trilinear", align_corners=False)            rho = F.interpolate(rho, size=target_size, mode="trilinear", align_corners=False)        return D, rhoclass SegmentationHead(nn.Module):    \"\"\"    Segmentation output head for BraTS pretraining.        Produces class probabilities for each voxel.    Uses the same decoder structure as the density decoder.    \"\"\"    def __init__(self, encoder_channels: dict, num_classes: int = 4,                 decoder_channels=(256, 128, 64, 32, 16), dropout: float = 0.1, deep_supervision: bool = True):        super().__init__()        enc = encoder_channels        dec = decoder_channels        self.bottleneck = ConvBlock3D(enc["layer4"], dec[0], dropout=dropout)        self.up1 = UpBlock3D(dec[0], enc["layer3"], dec[0], dropout=dropout)        self.up2 = UpBlock3D(dec[0], enc["layer2"], dec[1], dropout=dropout)        self.up3 = UpBlock3D(dec[1], enc["layer1"], dec[2], dropout=dropout)        self.up4 = UpBlock3D(dec[2], enc["stem"],   dec[3], dropout=dropout)        self.up5 = UpBlock3D(dec[3], 0,             dec[4], dropout=0.0)        self.final_conv = nn.Conv3d(dec[4], num_classes, kernel_size=1)                self.deep_supervision = deep_supervision        if self.deep_supervision:            self.ds2 = nn.Conv3d(dec[3], num_classes, kernel_size=1)            self.ds3 = nn.Conv3d(dec[2], num_classes, kernel_size=1)    def forward(self, features: dict, return_ds: bool = False):        \"\"\"        Returns:            Segmentation logits [B, num_classes, D, H, W] or list if return_ds.        \"\"\"        x = self.bottleneck(features["layer4"])        x = self.up1(x, features["layer3"])        x = self.up2(x, features["layer2"])        x = self.up3(x, features["layer1"])        ds3 = x        x = self.up4(x, features["stem"])        ds2 = x        x = self.up5(x)                out = self.final_conv(x)        if self.deep_supervision and return_ds:            return [out, self.ds2(ds2), self.ds3(ds3)]                    return out""")

In [ ]:
with open("models/hybrid_model.py", "w") as f:    f.write("""\"\"\"HybridTumorNet — The complete hybrid MONAI ResNet + Physics-Informed model.Architecture:    ┌───────────────────────────────────────────────────────────┐    │  Input: 4-channel MRI [B, 4, 128, 128, 128]              │    │         (T1, T1ce, T2, FLAIR)                             │    │                   │                                       │    │      ┌────────────▼────────────┐                          │    │      │   MONAI 3D ResNet-50    │                          │    │      │   (Encoder/Backbone)    │                          │    │      │      + Skip features    │                          │    │      └──┬──────┬──────┬───────┘                           │    │         │      │      │                                   │    │    ┌────▼──┐ ┌─▼────┐ ┌▼─────────┐                       │    │    │Density│ │Phys. │ │Segment.  │                        │    │    │Decoder│ │Params│ │Head      │                        │    │    │       │ │D(x),ρ│ │(pretrain)│                        │    │    └───┬───┘ └──┬───┘ └────┬─────┘                        │    │        │        │          │                               │    │   u(x,t)   D(x), ρ(x)   seg_logits                       │    │   [0,1]    positive      [num_cls]                        │    └───────────────────────────────────────────────────────────┘Output dict:    - "tumor_density": u(x,t) ∈ [0,1]   [B, 1, D, H, W]    - "diffusion":     D(x) > 0          [B, 1, D, H, W]    - "proliferation": ρ(x) > 0          [B, 1, D, H, W]    - "segmentation":  logits            [B, C, D, H, W]  (optional)\"\"\"import torchimport torch.nn as nnfrom .resnet_backbone import ResNetEncoderfrom .decoder import TumorDensityDecoder, PhysicsParameterHead, SegmentationHeadclass HybridTumorNet(nn.Module):    \"\"\"    Hybrid deep learning + physics-informed model for brain tumor analysis.        Combines:        1. MONAI 3D ResNet encoder for feature extraction from multi-modal MRI        2. Dense decoder for tumor density u(x,t) prediction        3. Physics parameter prediction (spatially varying D(x), ρ(x))        4. Optional segmentation head for BraTS pretraining        The model outputs are fed into physics-informed losses that enforce    the reaction–diffusion tumor growth equation.    \"\"\"    def __init__(self,                  resnet_variant: str = "resnet50",                 in_channels: int = 4,                 num_seg_classes: int = 4,                 pretrained_path: str = None,                 freeze_encoder_layers: int = 0,                 decoder_channels: tuple = (256, 128, 64, 32, 16),                 dropout: float = 0.1,                 use_seg_head: bool = True,                 predict_physics_params: bool = True,                 diffusion_range: tuple = (0.0001, 0.5),                 proliferation_range: tuple = (0.001, 0.5)):        \"\"\"        Args:            resnet_variant: Which ResNet to use ("resnet18", "resnet50", etc.).            in_channels: Number of MRI modalities (default 4).            num_seg_classes: Number of segmentation classes (BraTS = 4).            pretrained_path: Path to pretrained weight file (.pth).            freeze_encoder_layers: Number of encoder layers to freeze (0=none).            decoder_channels: Channel dimensions for decoder levels.            dropout: Dropout rate.            use_seg_head: Whether to include segmentation head.            predict_physics_params: Whether to predict D(x), ρ(x) spatially.            diffusion_range: (min, max) for diffusion coefficient.            proliferation_range: (min, max) for proliferation rate.        \"\"\"        super().__init__()        self.use_seg_head = use_seg_head        self.predict_physics_params = predict_physics_params        # === ENCODER ===        self.encoder = ResNetEncoder(            variant=resnet_variant,            in_channels=in_channels,            pretrained_path=pretrained_path,            freeze_layers=freeze_encoder_layers,        )        enc_channels = self.encoder.get_channels()        # === TUMOR DENSITY DECODER ===        self.density_decoder = TumorDensityDecoder(            encoder_channels=enc_channels,            decoder_channels=decoder_channels,            dropout=dropout,        )        # === PHYSICS PARAMETER HEAD ===        if predict_physics_params:            self.physics_head = PhysicsParameterHead(                encoder_channels=enc_channels,                diffusion_range=diffusion_range,                proliferation_range=proliferation_range,            )        else:            self.physics_head = None        # === SEGMENTATION HEAD (for pretraining) ===        if use_seg_head:            self.seg_head = SegmentationHead(                encoder_channels=enc_channels,                num_classes=num_seg_classes,                decoder_channels=decoder_channels,                dropout=dropout,            )        else:            self.seg_head = None    def forward(self, x: torch.Tensor, return_features: bool = False) -> dict:        \"\"\"        Forward pass through the hybrid model.        Args:            x: Input MRI tensor [B, 4, D, H, W].            return_features: If True, include encoder features in output.        Returns:            Dictionary with keys:                "tumor_density":  [B, 1, D, H, W] ∈ [0, 1]                "diffusion":      [B, 1, D, H, W] > 0  (if predict_physics_params)                "proliferation":  [B, 1, D, H, W] > 0  (if predict_physics_params)                "segmentation":   [B, C, D, H, W]       (if use_seg_head)                "features":       dict of encoder features (if return_features)        \"\"\"        # Encode        features = self.encoder(x)        output = {}                return_ds = self.training        # Tumor density prediction        tumor_density = self.density_decoder(features, return_ds=return_ds)        output["tumor_density"] = tumor_density        # Physics parameters        if self.predict_physics_params and self.physics_head is not None:            td_base = tumor_density[0] if isinstance(tumor_density, list) else tumor_density            target_size = td_base.shape[2:]  # (D, H, W)            D, rho = self.physics_head(features, target_size=target_size)            output["diffusion"] = D            output["proliferation"] = rho        # Segmentation        if self.use_seg_head and self.seg_head is not None:            seg_logits = self.seg_head(features, return_ds=return_ds)            output["segmentation"] = seg_logits        # Optional: return encoder features        if return_features:            output["features"] = features        return output    def get_parameter_groups(self, lr_backbone: float = 1e-5, lr_head: float = 1e-4):        \"\"\"        Get parameter groups with different learning rates.        Lower LR for pretrained backbone, higher for new heads.        Returns:            List of parameter group dicts for optimizer.        \"\"\"        backbone_params = list(self.encoder.parameters())        head_params = []        head_params.extend(list(self.density_decoder.parameters()))        if self.physics_head is not None:            head_params.extend(list(self.physics_head.parameters()))        if self.seg_head is not None:            head_params.extend(list(self.seg_head.parameters()))        return [            {"params": backbone_params, "lr": lr_backbone, "name": "backbone"},            {"params": head_params, "lr": lr_head, "name": "heads"},        ]    def freeze_encoder(self):        \"\"\"Freeze all encoder parameters.\"\"\"        for param in self.encoder.parameters():            param.requires_grad = False    def unfreeze_encoder(self):        \"\"\"Unfreeze all encoder parameters.\"\"\"        for param in self.encoder.parameters():            param.requires_grad = True    def set_phase(self, phase: str):        \"\"\"        Configure model for a specific training phase.        Args:            phase: "pretrain" (seg only), "finetune" (physics), or "joint" (all).        \"\"\"        if phase == "pretrain":            # Only segmentation head active, freeze physics parts            self.use_seg_head = True            if self.seg_head:                for p in self.seg_head.parameters():                    p.requires_grad = True            if self.physics_head:                for p in self.physics_head.parameters():                    p.requires_grad = False            for p in self.density_decoder.parameters():                p.requires_grad = False        elif phase == "finetune":            # Physics-informed training, freeze seg head            self.unfreeze_encoder()            for p in self.density_decoder.parameters():                p.requires_grad = True            if self.physics_head:                for p in self.physics_head.parameters():                    p.requires_grad = True            if self.seg_head:                for p in self.seg_head.parameters():                    p.requires_grad = False        elif phase == "joint":            # Everything trainable            self.unfreeze_encoder()            for p in self.parameters():                p.requires_grad = True        else:            raise ValueError(f"Unknown phase: {phase}")    @torch.no_grad()    def predict_tumor_growth(self, x: torch.Tensor,                               num_steps: int = 10,                              dt: float = 1.0,                              integration_method: str = "rk4") -> list:        \"\"\"        Predict tumor evolution over time using the learned parameters.                Uses RK4 or explicit Euler integration of the reaction-diffusion PDE:            ∂u/∂t = ∇·(D∇u) + ρu(1-u)        Args:            x: Input MRI [B, 4, D, H, W].            num_steps: Number of time steps to simulate.            dt: Time step size (days).            integration_method: "rk4" or "euler"        Returns:            List of tumor density fields [B, 1, D, H, W] at each time step.        \"\"\"        from utils.spatial_ops import SpatialGradients3D                self.eval()        output = self.forward(x)                u = output["tumor_density"]        D = output.get("diffusion", torch.full_like(u, 0.1))        rho = output.get("proliferation", torch.full_like(u, 0.025))        spatial_ops = SpatialGradients3D(dx=1.0, dy=1.0, dz=1.0).to(x.device)        def get_du_dt(u_current):            diffusion = spatial_ops.divergence_of_flux(u_current, D)            reaction = rho * u_current * (1.0 - u_current)            return diffusion + reaction        trajectory = [u.clone()]        for step in range(num_steps):            if integration_method == "rk4":                k1 = get_du_dt(u)                k2 = get_du_dt(u + 0.5 * dt * k1)                k3 = get_du_dt(u + 0.5 * dt * k2)                k4 = get_du_dt(u + dt * k3)                u = u + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)            else:                # Euler step                u = u + dt * get_du_dt(u)                        # Enforce constraints            u = torch.clamp(u, 0.0, 1.0)                        trajectory.append(u.clone())        return trajectory    def count_parameters(self) -> dict:        \"\"\"Count trainable and total parameters.\"\"\"        total = sum(p.numel() for p in self.parameters())        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)                encoder_params = sum(p.numel() for p in self.encoder.parameters())        decoder_params = sum(p.numel() for p in self.density_decoder.parameters())                info = {            "total": total,            "trainable": trainable,            "frozen": total - trainable,            "encoder": encoder_params,            "density_decoder": decoder_params,        }                if self.physics_head:            info["physics_head"] = sum(p.numel() for p in self.physics_head.parameters())        if self.seg_head:            info["seg_head"] = sum(p.numel() for p in self.seg_head.parameters())                return info""")

In [ ]:
with open("data/preprocessing.py", "w") as f:    f.write("""\"\"\"MRI preprocessing and augmentation transforms using MONAI.Pipeline:    1. Load NIfTI files (4 modalities + segmentation mask)    2. Reorient to RAS+    3. Resample to uniform spacing    4. Crop to brain region / fixed ROI    5. Intensity normalization (z-score per modality within brain mask)    6. Random augmentations (train only)\"\"\"from monai import transforms as Tdef get_train_transforms(spatial_size=(128, 128, 128),                           intensity_shift=0.1,                          intensity_scale=0.1):    \"\"\"    Get training transforms with data augmentation.    Returns:        MONAI Compose transform pipeline.    \"\"\"    return T.Compose([        # --- Load ---        T.LoadImaged(keys=["image", "label"], ensure_channel_first=True),        # --- Orientation & Spacing ---        T.Orientationd(keys=["image", "label"], axcodes="RAS"),        T.Spacingd(            keys=["image", "label"],            pixdim=(1.0, 1.0, 1.0),            mode=("bilinear", "nearest"),        ),        # --- Intensity normalization (z-score per channel) ---        T.NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),        # --- Crop / Resize ---        T.CropForegroundd(keys=["image", "label"], source_key="image", margin=10),        T.SpatialPadd(keys=["image", "label"], spatial_size=spatial_size),        T.RandSpatialCropd(            keys=["image", "label"],            roi_size=spatial_size,            random_center=True,            random_size=False,        ),        # --- Augmentation ---        T.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),        T.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),        T.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),        T.RandRotate90d(keys=["image", "label"], prob=0.3, max_k=3),        T.RandScaleIntensityd(keys=["image"], factors=intensity_scale, prob=0.5),        T.RandShiftIntensityd(keys=["image"], offsets=intensity_shift, prob=0.5),        T.RandGaussianNoised(keys=["image"], std=0.05, prob=0.3),        T.RandGaussianSmoothd(            keys=["image"],            sigma_x=(0.5, 1.15),            sigma_y=(0.5, 1.15),            sigma_z=(0.5, 1.15),            prob=0.2,        ),        # --- Convert labels ---        # BraTS labels: 0, 1, 2, 4 → remap 4→3        T.MapLabelValued(keys=["label"], orig_labels=[0, 1, 2, 4], target_labels=[0, 1, 2, 3]),        # --- To Tensor ---        T.EnsureTyped(keys=["image", "label"], dtype="float32"),    ])def get_val_transforms(spatial_size=(128, 128, 128)):    \"\"\"    Get validation/test transforms (no augmentation).    Returns:        MONAI Compose transform pipeline.    \"\"\"    return T.Compose([        # --- Load ---        T.LoadImaged(keys=["image", "label"], ensure_channel_first=True),        # --- Orientation & Spacing ---        T.Orientationd(keys=["image", "label"], axcodes="RAS"),        T.Spacingd(            keys=["image", "label"],            pixdim=(1.0, 1.0, 1.0),            mode=("bilinear", "nearest"),        ),        # --- Intensity normalization ---        T.NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),        # --- Crop / Resize (deterministic) ---        T.CropForegroundd(keys=["image", "label"], source_key="image", margin=10),        T.SpatialPadd(keys=["image", "label"], spatial_size=spatial_size),        T.CenterSpatialCropd(keys=["image", "label"], roi_size=spatial_size),        # --- Convert labels ---        T.MapLabelValued(keys=["label"], orig_labels=[0, 1, 2, 4], target_labels=[0, 1, 2, 3]),        # --- To Tensor ---        T.EnsureTyped(keys=["image", "label"], dtype="float32"),    ])def get_inference_transforms(spatial_size=(128, 128, 128)):    \"\"\"    Transforms for inference (no label loading).    Returns:        MONAI Compose transform pipeline.    \"\"\"    return T.Compose([        T.LoadImaged(keys=["image"], ensure_channel_first=True),        T.Orientationd(keys=["image"], axcodes="RAS"),        T.Spacingd(keys=["image"], pixdim=(1.0, 1.0, 1.0), mode="bilinear"),        T.NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),        T.CropForegroundd(keys=["image"], source_key="image", margin=10),        T.SpatialPadd(keys=["image"], spatial_size=spatial_size),        T.CenterSpatialCropd(keys=["image"], roi_size=spatial_size),        T.EnsureTyped(keys=["image"], dtype="float32"),    ])def get_longitudinal_transforms(spatial_size=(128, 128, 128)):    \"\"\"    Transforms for longitudinal (paired) MRI data.    Loads two time-point images with the same spatial transforms.        Keys: "image_t1", "image_t2", "label_t1", "label_t2"    \"\"\"    return T.Compose([        # --- Load all ---        T.LoadImaged(            keys=["image_t1", "image_t2", "label_t1", "label_t2"],            ensure_channel_first=True,        ),        T.Orientationd(            keys=["image_t1", "image_t2", "label_t1", "label_t2"],            axcodes="RAS",        ),        T.Spacingd(            keys=["image_t1", "image_t2", "label_t1", "label_t2"],            pixdim=(1.0, 1.0, 1.0),            mode=("bilinear", "bilinear", "nearest", "nearest"),        ),        # --- Normalize ---        T.NormalizeIntensityd(keys=["image_t1", "image_t2"], nonzero=True, channel_wise=True),        # --- Crop & Pad (use t1 as reference) ---        T.CropForegroundd(            keys=["image_t1", "image_t2", "label_t1", "label_t2"],            source_key="image_t1",            margin=10,        ),        T.SpatialPadd(            keys=["image_t1", "image_t2", "label_t1", "label_t2"],            spatial_size=spatial_size,        ),        T.CenterSpatialCropd(            keys=["image_t1", "image_t2", "label_t1", "label_t2"],            roi_size=spatial_size,        ),        # --- Labels ---        T.MapLabelValued(            keys=["label_t1", "label_t2"],            orig_labels=[0, 1, 2, 4],            target_labels=[0, 1, 2, 3],        ),        T.EnsureTyped(            keys=["image_t1", "image_t2", "label_t1", "label_t2"],            dtype="float32",        ),    ])""")

In [ ]:
with open("data/synthetic_longitudinal.py", "w") as f:    f.write("""\"\"\"Synthetic Longitudinal Data Generator for Physics-Constrained Training.PURPOSE & SCIENTIFIC JUSTIFICATION------------------------------------BraTS 2021 is a CROSS-SECTIONAL dataset (single time point per patient).The Fisher-KPP PDE residual requires temporal derivatives:    ∂u/∂t ≈ (u(t₂) - u(t₁)) / ΔtTo supply this without real longitudinal MRI pairs, we use a well-establishedmethod from the computational oncology literature (Mang et al. 2019,Lipkova et al. 2019):    1. Treat the segmentation-derived density u₀ as the INITIAL condition u(t₁)    2. Numerically integrate the Fisher-KPP PDE forward by Δt days       using default/estimated diffusion and proliferation parameters    3. The numerically simulated u(t₂) serves as the synthetic ground truthThis is scientifically valid because:    a) The PDE model is self-consistent — we train the network to predict       parameters (D, ρ) that reproduce the numerically integrated evolution    b) It constrains the network to learn biophysically plausible parameters    c) It is clearly labeled as SYNTHETIC in all outputsThis approach is DIFFERENT from claiming real growth was observed.The distinction must be stated clearly in any paper.References:    - Mang et al. (2019) CLAIRE: A distributed-memory solver for      constrained large diffeomorphic image registration. SIAM J. Sci. Comput.    - Lipkova et al. (2019) Personalized Radiotherapy Design for Glioblastoma.      IEEE Trans. Med. Imag.    - Swanson et al. (2000) A mathematical modelling tool for predicting      survival of individual patients following resection of glioblastoma.      Brit. J. Cancer.\"\"\"import numpy as npimport torchimport torch.nn.functional as Ffrom typing import Tuple, Optionalclass FisherKPPSolver:    \"\"\"    Numerically solves the Fisher-KPP reaction-diffusion PDE:        ∂u/∂t = ∇·(D(x)∇u) + ρ·u·(1 - u)    using explicit 4th-order Runge-Kutta integration on a 3D voxel grid.    This solver is used to generate SYNTHETIC longitudinal tumor density    fields from a single BraTS time point. The output is the SIMULATED    tumor state Δt days in the future.    Stability constraint (CFL condition for explicit solver):        Δt_step ≤ (Δx)² / (6 · D_max)        For Δx=1mm, D_max=0.1: Δt_step ≤ 1.67 days    Parameters are intentionally set conservatively to avoid numerical blowup.    \"\"\"    def __init__(        self,        dx: float = 1.0,        dy: float = 1.0,        dz: float = 1.0,        default_D: float = 0.05,       # mm²/day — conservative default        default_rho: float = 0.02,     # 1/day — proliferation rate        max_dt_step: float = 1.0,      # days per RK4 step (CFL-safe at D=0.05)        device: str = "cpu",    ):        \"\"\"        Args:            dx, dy, dz: Voxel spacing in mm along each axis.            default_D: Default diffusion coefficient (mm²/day).            default_rho: Default proliferation rate (1/day).            max_dt_step: Maximum time step per RK4 integration step (days).                         Ensure CFL: max_dt_step < dx² / (6 * default_D)            device: Torch device string.        \"\"\"        self.dx = dx        self.dy = dy        self.dz = dz        self.default_D = default_D        self.default_rho = default_rho        self.max_dt_step = max_dt_step        self.device = torch.device(device)        # CFL stability check        cfl_limit = min(dx, dy, dz) ** 2 / (6.0 * default_D + 1e-9)        if max_dt_step > cfl_limit:            import warnings            warnings.warn(                f"max_dt_step={max_dt_step} may violate CFL condition "                f"(limit={cfl_limit:.3f} for D={default_D}). "                f"Consider reducing max_dt_step for numerical stability.",                RuntimeWarning,            )        # Pre-build finite-difference kernels (fixed, non-trainable)        self._build_kernels()    def _build_kernels(self):        \"\"\"Build 3D finite-difference convolution kernels.\"\"\"        # 7-point Laplacian stencil        k_lap = torch.zeros(1, 1, 3, 3, 3, dtype=torch.float32)        k_lap[0, 0, 1, 1, 1] = -(2 / self.dx**2 + 2 / self.dy**2 + 2 / self.dz**2)        k_lap[0, 0, 0, 1, 1] = 1.0 / self.dx**2        k_lap[0, 0, 2, 1, 1] = 1.0 / self.dx**2        k_lap[0, 0, 1, 0, 1] = 1.0 / self.dy**2        k_lap[0, 0, 1, 2, 1] = 1.0 / self.dy**2        k_lap[0, 0, 1, 1, 0] = 1.0 / self.dz**2        k_lap[0, 0, 1, 1, 2] = 1.0 / self.dz**2        # Central difference kernels for ∂/∂x, ∂/∂y, ∂/∂z        k_dx = torch.zeros(1, 1, 3, 3, 3, dtype=torch.float32)        k_dx[0, 0, 0, 1, 1] = -1.0 / (2 * self.dx)        k_dx[0, 0, 2, 1, 1] =  1.0 / (2 * self.dx)        k_dy = torch.zeros(1, 1, 3, 3, 3, dtype=torch.float32)        k_dy[0, 0, 1, 0, 1] = -1.0 / (2 * self.dy)        k_dy[0, 0, 1, 2, 1] =  1.0 / (2 * self.dy)        k_dz = torch.zeros(1, 1, 3, 3, 3, dtype=torch.float32)        k_dz[0, 0, 1, 1, 0] = -1.0 / (2 * self.dz)        k_dz[0, 0, 1, 1, 2] =  1.0 / (2 * self.dz)        self.k_lap = k_lap.to(self.device)        self.k_dx  = k_dx.to(self.device)        self.k_dy  = k_dy.to(self.device)        self.k_dz  = k_dz.to(self.device)    def _rhs(        self,        u: torch.Tensor,        D: torch.Tensor,        rho: torch.Tensor,    ) -> torch.Tensor:        \"\"\"        Compute Fisher-KPP PDE right-hand side: ∇·(D∇u) + ρ·u·(1-u).        Uses product rule: ∇·(D∇u) = D·∇²u + ∇D·∇u        Args:            u:   [B, 1, D, H, W] tumor density ∈ [0, 1]            D:   [B, 1, D, H, W] diffusion coefficient (mm²/day)            rho: [B, 1, D, H, W] or scalar proliferation rate (1/day)        Returns:            [B, 1, D, H, W] — du/dt        \"\"\"        # Laplacian of u        lap_u = F.conv3d(u, self.k_lap, padding=1)        # Gradient of u        du_dx = F.conv3d(u, self.k_dx, padding=1)        du_dy = F.conv3d(u, self.k_dy, padding=1)        du_dz = F.conv3d(u, self.k_dz, padding=1)        # Gradient of D        dD_dx = F.conv3d(D, self.k_dx, padding=1)        dD_dy = F.conv3d(D, self.k_dy, padding=1)        dD_dz = F.conv3d(D, self.k_dz, padding=1)        # Divergence of flux: ∇·(D∇u) = D·∇²u + ∇D·∇u        diffusion = D * lap_u + (dD_dx * du_dx + dD_dy * du_dy + dD_dz * du_dz)        # Reaction term        reaction = rho * u * (1.0 - u)        return diffusion + reaction    @torch.no_grad()    def simulate(        self,        u0: torch.Tensor,        delta_t_days: float,        D: Optional[torch.Tensor] = None,        rho: Optional[torch.Tensor] = None,        brain_mask: Optional[torch.Tensor] = None,        add_noise: bool = True,        noise_std: float = 0.005,    ) -> Tuple[torch.Tensor, torch.Tensor]:        \"\"\"        Simulate tumor growth for delta_t_days using RK4.        Args:            u0: Initial tumor density [B, 1, D, H, W] ∈ [0, 1].            delta_t_days: Total simulation time in days.            D: Spatially-varying diffusion [B, 1, D, H, W]. If None, uses default.            rho: Proliferation rate [B, 1, D, H, W] or scalar. If None, uses default.            brain_mask: Binary brain mask [B, 1, D, H, W]. Enforces no-flux BC.            add_noise: Whether to add small realistic measurement noise to u_t2.            noise_std: Standard deviation of measurement noise.        Returns:            (u_t2, du_dt):                u_t2:  [B, 1, D, H, W] — synthetic tumor state at t₂                du_dt: [B, 1, D, H, W] — finite-difference temporal derivative        \"\"\"        u0 = u0.to(self.device).float()        if D is None:            D = torch.full_like(u0, self.default_D)        else:            D = D.to(self.device).float()        if rho is None:            rho = torch.full_like(u0, self.default_rho)        else:            rho = rho.to(self.device).float()        # Number of integration steps (ensure CFL)        n_steps = max(1, int(np.ceil(delta_t_days / self.max_dt_step)))        dt_step = delta_t_days / n_steps        u = u0.clone()        for _ in range(n_steps):            # RK4 integration            k1 = self._rhs(u, D, rho)            k2 = self._rhs(torch.clamp(u + 0.5 * dt_step * k1, 0.0, 1.0), D, rho)            k3 = self._rhs(torch.clamp(u + 0.5 * dt_step * k2, 0.0, 1.0), D, rho)            k4 = self._rhs(torch.clamp(u + dt_step * k3,        0.0, 1.0), D, rho)            u = u + (dt_step / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)            u = torch.clamp(u, 0.0, 1.0)            # Apply brain mask (no-flux: tumor stays inside brain)            if brain_mask is not None:                u = u * brain_mask        u_t2 = u        # Add small realistic measurement noise to simulate MRI variability        if add_noise and noise_std > 0.0:            noise = torch.randn_like(u_t2) * noise_std            u_t2 = torch.clamp(u_t2 + noise, 0.0, 1.0)        # Temporal derivative (first-order finite difference)        du_dt = (u_t2 - u0) / (delta_t_days + 1e-9)        return u_t2, du_dtdef generate_synthetic_pair(    seg_label: torch.Tensor,    delta_t_days: float = 90.0,    D_wm: float = 0.08,    D_gm: float = 0.016,    rho: float = 0.02,    brain_mask: Optional[torch.Tensor] = None,    device: str = "cpu",) -> dict:    \"\"\"    High-level API: Generate a synthetic longitudinal pair from a BraTS    segmentation mask using the Fisher-KPP PDE.    Scientific basis:        - D_wm (white matter diffusion) ≈ 5x D_gm (Harpold et al. 2007)        - Default D and ρ from Swanson et al. (2000) glioma growth model        - Δt = 90 days ≈ typical BraTS re-scan interval    Args:        seg_label: Segmentation [B, 1, D, H, W] with BraTS labels {0,1,2,3}.        delta_t_days: Simulation time (days between virtual scans).        D_wm: White-matter diffusion coefficient (mm²/day).        D_gm: Grey-matter diffusion coefficient (mm²/day).        rho: Proliferation rate (1/day).        brain_mask: Binary brain mask [B, 1, D, H, W].        device: Torch device string.    Returns:        dict with keys:            "u_t1":    Initial tumor density [B, 1, D, H, W]            "u_t2":    Simulated tumor density at t₂ [B, 1, D, H, W]            "du_dt":   Temporal derivative (u_t2 - u_t1) / Δt [B, 1, D, H, W]            "D_field": Diffusion field used [B, 1, D, H, W]            "rho_field": Proliferation field [B, 1, D, H, W]            "delta_t": Simulation time (days)            "is_synthetic": True (always — must be stated in paper)    \"\"\"    seg = seg_label.to(device).float()    # --- Construct initial tumor density from segmentation ---    # Biophysically grounded density mapping:    #   ET  (class 3) → u = 1.0  (highest cell density, contrast-enhancing)    #   NCR (class 1) → u = 0.6  (necrotic core, cell debris)    #   ED  (class 2) → u = 0.2  (infiltrative edema, sparse invasion)    #   BG  (class 0) → u = 0.0    u_t1 = torch.zeros_like(seg)    u_t1 = torch.where(seg == 2, torch.tensor(0.2, device=device), u_t1)    u_t1 = torch.where(seg == 1, torch.tensor(0.6, device=device), u_t1)    u_t1 = torch.where(seg == 3, torch.tensor(1.0, device=device), u_t1)    # --- Construct spatially varying diffusion field ---    # Approximate tissue type from MRI: use segmentation neighborhood    # as a proxy — inside tumor use D_gm, periphery use D_wm    # This is a simplification; ideally requires T1 white-matter segmentation.    # Use binary dilation to estimate infiltration zone (white matter region)    if brain_mask is not None:        # Region inside brain but outside tumor core → assume mix, use D_wm weighted        tumor_core = (seg > 0).float()        brain_region = brain_mask.float()        perilesional = brain_region * (1.0 - tumor_core)  # brain minus tumor        D_field = D_gm * tumor_core + D_wm * perilesional    else:        D_field = torch.full_like(u_t1, D_wm)  # Uniform fallback    D_field = D_field.to(device)    rho_field = torch.full_like(u_t1, rho)    # --- Simulate growth ---    solver = FisherKPPSolver(        default_D=D_wm,        default_rho=rho,        max_dt_step=1.0,        device=device,    )    u_t2, du_dt = solver.simulate(        u0=u_t1,        delta_t_days=delta_t_days,        D=D_field,        rho=rho_field,        brain_mask=brain_mask,        add_noise=True,        noise_std=0.005,    )    return {        "u_t1": u_t1,        "u_t2": u_t2,        "du_dt": du_dt,        "D_field": D_field,        "rho_field": rho_field,        "delta_t": delta_t_days,        "is_synthetic": True,  # MUST be stated in paper    }if __name__ == "__main__":    \"\"\"Quick test / demo of the synthetic longitudinal generator.\"\"\"    print("Testing FisherKPPSolver...")    # Create a simple spherical tumor density    B, D, H, W = 1, 64, 64, 64    u0 = torch.zeros(B, 1, D, H, W)    # Small sphere at center    for i in range(D):        for j in range(H):            for k in range(W):                r = ((i - D//2)**2 + (j - H//2)**2 + (k - W//2)**2) ** 0.5                if r < 5:                    u0[0, 0, i, j, k] = 1.0                elif r < 8:                    u0[0, 0, i, j, k] = 0.5    D_field = torch.full_like(u0, 0.05)    rho_field = torch.full_like(u0, 0.02)    brain_mask = torch.ones_like(u0)    solver = FisherKPPSolver(default_D=0.05, default_rho=0.02, max_dt_step=1.0)    u_t2, du_dt = solver.simulate(u0, delta_t_days=30.0, D=D_field, rho=rho_field,                                   brain_mask=brain_mask)    print(f"  u0   range: [{u0.min():.4f}, {u0.max():.4f}]")    print(f"  u_t2 range: [{u_t2.min():.4f}, {u_t2.max():.4f}]")    print(f"  du_dt range: [{du_dt.min():.4f}, {du_dt.max():.4f}]")    # Tumor should have grown (u_t2 > u_t1 on average in tumor region)    tumor_mask = (u0 > 0.1).float()    mean_u0   = (u0   * tumor_mask).sum() / tumor_mask.sum().clamp(min=1)    mean_u_t2 = (u_t2 * tumor_mask).sum() / tumor_mask.sum().clamp(min=1)    print(f"  Mean density in tumor: {mean_u0:.4f} → {mean_u_t2:.4f} (should be ≥)")    assert mean_u_t2 >= mean_u0 * 0.95, "Tumor should not shrink with positive ρ!"    print("✓ FisherKPPSolver test PASSED")""")

In [ ]:
with open("data/dataset.py", "w") as f:    f.write("""\"\"\"BraTS Dataset loaders.Supports:    - Standard BraTS 2021 layout (single time-point per patient)    - Longitudinal pairs (for patients with multiple scans)    - Automatic train/val split    - Missing modality handling\"\"\"import osimport globimport jsonimport numpy as npimport torchfrom torch.utils.data import Dataset, DataLoaderfrom monai.data import CacheDataset, SmartCacheDataset, list_data_collatefrom .preprocessing import (    get_train_transforms,    get_val_transforms,    get_longitudinal_transforms,)def discover_brats_data(data_root: str, modalities=("t1", "t1ce", "t2", "flair")):    \"\"\"    Auto-discover BraTS dataset structure.    Expected layout:        data_root/        ├── BraTS2021_00000/        │   ├── BraTS2021_00000_t1.nii.gz        │   ├── BraTS2021_00000_t1ce.nii.gz        │   ├── BraTS2021_00000_t2.nii.gz        │   ├── BraTS2021_00000_flair.nii.gz        │   └── BraTS2021_00000_seg.nii.gz        ├── BraTS2021_00001/        ...    Returns:        List of dicts with keys "image" (list of 4 paths) and "label" (seg path).    \"\"\"    data_dicts = []    patient_dirs = sorted(glob.glob(os.path.join(data_root, "BraTS*")))    if not patient_dirs:        # Try alternative naming        patient_dirs = sorted(glob.glob(os.path.join(data_root, "*")))        patient_dirs = [d for d in patient_dirs if os.path.isdir(d)]    for patient_dir in patient_dirs:        patient_id = os.path.basename(patient_dir)                # Find modality files        mod_files = []        found_all = True        for mod in modalities:            pattern = os.path.join(patient_dir, f"*_{mod}.nii.gz")            matches = glob.glob(pattern)            if not matches:                # Try alternative naming                pattern = os.path.join(patient_dir, f"*{mod}*.nii.gz")                matches = glob.glob(pattern)                        if matches:                mod_files.append(matches[0])            else:                found_all = False                print(f"Warning: Missing {mod} for {patient_id}")                break        # Find segmentation        seg_pattern = os.path.join(patient_dir, "*_seg.nii.gz")        seg_matches = glob.glob(seg_pattern)        if found_all and seg_matches:            data_dicts.append({                "image": mod_files,  # List of 4 modality paths                "label": seg_matches[0],                "patient_id": patient_id,            })    print(f"Found {len(data_dicts)} patients with all modalities")    return data_dictsdef discover_msd_data(data_root: str):    \"\"\"    Auto-discover Medical Segmentation Decathlon (MSD) dataset structure.    Expected layout:        data_root/        ├── imagesTr/        │   ├── BRATS_001.nii.gz        │   ...        ├── labelsTr/        │   ├── BRATS_001.nii.gz        │   ...    Returns:        List of dicts with keys "image" and "label" pointing to the paired files.    \"\"\"    data_dicts = []    images_dir = os.path.join(data_root, "imagesTr")    labels_dir = os.path.join(data_root, "labelsTr")    image_files = sorted(glob.glob(os.path.join(images_dir, "*.nii.gz")))        for img_path in image_files:        basename = os.path.basename(img_path)        label_path = os.path.join(labels_dir, basename)                if os.path.exists(label_path):            data_dicts.append({                "image": img_path,  # single 4D nifti                "label": label_path,                "patient_id": basename.replace(".nii.gz", ""),            })    print(f"Found {len(data_dicts)} MSD patients with paired labels")    return data_dictsclass BraTSDataset:    \"\"\"    BraTS dataset manager. Creates MONAI CacheDatasets with    appropriate transforms for training and validation.    \"\"\"    def __init__(self, data_root: str, spatial_size=(128, 128, 128),                 val_split: float = 0.2, cache_rate: float = 0.5,                 seed: int = 42):        \"\"\"        Args:            data_root: Path to BraTS data directory.            spatial_size: Target volume dimensions.            val_split: Fraction of data for validation.            cache_rate: Fraction of data to cache in memory (0.0 to 1.0).            seed: Random seed for reproducible splits.        \"\"\"        self.data_root = data_root        self.spatial_size = spatial_size        self.val_split = val_split        self.cache_rate = cache_rate        # Discover data        if os.path.exists(os.path.join(data_root, "imagesTr")) and os.path.exists(os.path.join(data_root, "labelsTr")):            self.all_data = discover_msd_data(data_root)        else:            self.all_data = discover_brats_data(data_root)        if len(self.all_data) == 0:            raise RuntimeError(                f"No BraTS data found in {data_root}. "                "Check directory structure. Expected: data_root/BraTS2021_XXXXX/"            )        # Split        np.random.seed(seed)        indices = np.random.permutation(len(self.all_data))        val_size = int(len(indices) * val_split)        val_indices = indices[:val_size]        train_indices = indices[val_size:]        self.train_data = [self.all_data[i] for i in train_indices]        self.val_data = [self.all_data[i] for i in val_indices]        print(f"Train: {len(self.train_data)}, Val: {len(self.val_data)}")    def get_train_dataset(self):        \"\"\"Create cached training dataset.\"\"\"        return CacheDataset(            data=self.train_data,            transform=get_train_transforms(self.spatial_size),            cache_rate=self.cache_rate,            num_workers=4,        )    def get_val_dataset(self):        \"\"\"Create cached validation dataset.\"\"\"        return CacheDataset(            data=self.val_data,            transform=get_val_transforms(self.spatial_size),            cache_rate=1.0,  # Cache all validation data            num_workers=4,        )class LongitudinalBraTSDataset(Dataset):    \"\"\"    Dataset for longitudinal (paired) brain MRI scans.        Each sample contains two time-point scans from the same patient,    used for computing temporal derivatives ∂u/∂t for the physics loss.        Expected layout:        data_root/        ├── patient_001/        │   ├── timepoint_1/        │   │   ├── t1.nii.gz, t1ce.nii.gz, t2.nii.gz, flair.nii.gz, seg.nii.gz        │   └── timepoint_2/        │       ├── t1.nii.gz, t1ce.nii.gz, t2.nii.gz, flair.nii.gz, seg.nii.gz        ...        Or provide a JSON manifest:    [        {            "patient_id": "001",            "timepoint_1": {"image": [...], "label": "...", "days": 0},            "timepoint_2": {"image": [...], "label": "...", "days": 90}        }    ]    \"\"\"    def __init__(self, manifest_path: str = None, data_root: str = None,                 spatial_size=(128, 128, 128), transform=None):        self.spatial_size = spatial_size        self.pairs = []        if manifest_path and os.path.exists(manifest_path):            with open(manifest_path) as f:                self.pairs = json.load(f)        elif data_root:            self.pairs = self._discover_longitudinal(data_root)        self.transform = transform or get_longitudinal_transforms(spatial_size)        print(f"Found {len(self.pairs)} longitudinal pairs")    def _discover_longitudinal(self, data_root):        \"\"\"Auto-discover longitudinal data structure.\"\"\"        pairs = []        patient_dirs = sorted(glob.glob(os.path.join(data_root, "*")))                for pdir in patient_dirs:            if not os.path.isdir(pdir):                continue                        timepoints = sorted(glob.glob(os.path.join(pdir, "timepoint_*")))            if len(timepoints) < 2:                continue                        # Create pairs from consecutive timepoints            for i in range(len(timepoints) - 1):                tp1_mods = sorted(glob.glob(os.path.join(timepoints[i], "*.nii.gz")))                tp2_mods = sorted(glob.glob(os.path.join(timepoints[i + 1], "*.nii.gz")))                                if len(tp1_mods) >= 5 and len(tp2_mods) >= 5:  # 4 modalities + seg                    pairs.append({                        "patient_id": os.path.basename(pdir),                        "image_t1": [f for f in tp1_mods if "seg" not in f][:4],                        "image_t2": [f for f in tp2_mods if "seg" not in f][:4],                        "label_t1": [f for f in tp1_mods if "seg" in f][0],                        "label_t2": [f for f in tp2_mods if "seg" in f][0],                        "delta_t": 90.0,  # Default days between scans                    })        return pairs    def __len__(self):        return len(self.pairs)    def __getitem__(self, idx):        pair = self.pairs[idx]        sample = {            "image_t1": pair["image_t1"],            "image_t2": pair["image_t2"],            "label_t1": pair["label_t1"],            "label_t2": pair["label_t2"],        }        if self.transform:            sample = self.transform(sample)        sample["delta_t"] = torch.tensor(pair.get("delta_t", 90.0), dtype=torch.float32)        sample["patient_id"] = pair["patient_id"]        return sampledef get_train_val_dataloaders(data_root: str, batch_size: int = 2,                               spatial_size=(128, 128, 128),                               num_workers: int = 4,                               cache_rate: float = 0.5,                               val_split: float = 0.2,                               pin_memory: bool = True):    \"\"\"    Create training and validation DataLoaders.    Returns:        (train_loader, val_loader)    \"\"\"    dataset = BraTSDataset(        data_root=data_root,        spatial_size=spatial_size,        val_split=val_split,        cache_rate=cache_rate,    )    train_loader = DataLoader(        dataset.get_train_dataset(),        batch_size=batch_size,        shuffle=True,        num_workers=num_workers,        pin_memory=pin_memory,        collate_fn=list_data_collate,        drop_last=True,    )    val_loader = DataLoader(        dataset.get_val_dataset(),        batch_size=1,        shuffle=False,        num_workers=num_workers,        pin_memory=pin_memory,        collate_fn=list_data_collate,    )    return train_loader, val_loader""")

In [ ]:
with open("losses/physics_loss.py", "w") as f:    f.write("""\"\"\"Physics-Informed Loss Functions for Tumor Growth.Implements the PDE residual and constraint losses:    L_PDE = ||∂u/∂t − ∇·(D(x)∇u) − ρu(1−u)||²     (Fisher-KPP)    L_PDE = ||∂u/∂t − ∇·(D(x)∇u) − ρu·ln(K/(u+ε))||²  (Gompertz)        L_IC  = ||u(x, t₀) − u₀(x)||²                    (Initial condition)    L_BC  = ||∇u · n||²  at boundaries                 (No-flux Neumann BC)    L_reg = λ_pos·||ReLU(−u)||² + λ_bound·||ReLU(u−1)||² + λ_smooth·||∇u||²All losses work on 3D volume tensors [B, 1, D, H, W].\"\"\"import torchimport torch.nn as nnimport torch.nn.functional as Fimport sysimport ossys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))from utils.spatial_ops import SpatialGradients3D, compute_temporal_derivativeclass PDEResidualLoss(nn.Module):    \"\"\"    PDE residual loss enforcing the Fisher-KPP reaction-diffusion equation.    Residual R = ∂u/∂t − ∇·(D(x)∇u) − ρ·u·(1 − u)    Loss: L_PDE = (1/N) Σ R²  [mean squared residual over domain]    Temporal derivative (∂u/∂t):        - If du_dt is provided (from synthetic longitudinal pairs): used directly.        - If du_dt is None: quasi-steady-state mode — penalizes |RHS| directly.          This is valid when the tumor is assumed to be at a slowly-changing state.          NOTE: quasi-steady-state is a WEAKER constraint — state clearly in paper.    Tissue-aware diffusion (Harpold et al. 2007):        If tissue_mask is provided, D(x) is scaled:            D_effective(x) = D_pred(x) * [D_wm in WM, D_gm in GM]        This ensures biophysically realistic diffusion anisotropy.    \"\"\"    def __init__(        self,        pde_model: str = "fisher_kpp",        dx: float = 1.0,        dy: float = 1.0,        dz: float = 1.0,        carrying_capacity: float = 1.0,        epsilon: float = 1e-7,        D_white_matter: float = 0.08,        D_grey_matter: float = 0.016,    ):        \"\"\"        Args:            pde_model: "fisher_kpp" or "gompertz".            dx, dy, dz: Voxel spacing in mm.            carrying_capacity: K in Gompertz model.            epsilon: Small constant for numerical stability.            D_white_matter: Baseline WM diffusion (mm²/day). Harpold et al. 2007.            D_grey_matter: Baseline GM diffusion (mm²/day). Harpold et al. 2007.        \"\"\"        super().__init__()        self.pde_model = pde_model        self.K = carrying_capacity        self.eps = epsilon        self.D_wm = D_white_matter        self.D_gm = D_grey_matter        self.spatial_ops = SpatialGradients3D(dx, dy, dz)    def _apply_tissue_diffusion(        self,        D: torch.Tensor,        tissue_mask: torch.Tensor,    ) -> torch.Tensor:        \"\"\"        Scale predicted D(x) by tissue-specific baseline values.        Strategy: The network learns a RELATIVE diffusion field D_pred ∈ (0,1].        We then scale it: D_effective = D_pred * D_tissue(x)        where D_tissue = D_wm inside white matter, D_gm otherwise.        Args:            D: Predicted diffusion [B, 1, D, H, W] (positive, raw network output).            tissue_mask: White-matter probability map [B, 1, D, H, W] ∈ [0, 1].                         1.0 = fully white matter, 0.0 = grey matter / background.        Returns:            D_effective [B, 1, D, H, W] with tissue-appropriate magnitude.        \"\"\"        D_tissue = tissue_mask * self.D_wm + (1.0 - tissue_mask) * self.D_gm        return D * D_tissue    def forward(        self,        u: torch.Tensor,        D: torch.Tensor,        rho: torch.Tensor,        du_dt: torch.Tensor = None,        mask: torch.Tensor = None,        tissue_mask: torch.Tensor = None,    ) -> torch.Tensor:        \"\"\"        Compute the PDE residual loss.        Args:            u: Tumor density [B, 1, D, H, W] ∈ [0, 1].            D: Predicted diffusion coefficient [B, 1, D, H, W] > 0.            rho: Proliferation rate [B, 1, D, H, W] or scalar tensor.            du_dt: Temporal derivative [B, 1, D, H, W] (from synthetic pairs                   or real longitudinal data). If None, quasi-steady-state mode.            mask: Optional brain mask [B, 1, D, H, W]. Loss computed only in mask.            tissue_mask: Optional white-matter map [B, 1, D, H, W] ∈ [0,1].                         If provided, applies tissue-aware diffusion scaling.        Returns:            Scalar PDE residual loss (non-negative).        \"\"\"        # Move spatial ops to same device        self.spatial_ops = self.spatial_ops.to(u.device)        # Apply tissue-aware diffusion scaling if white-matter mask provided        if tissue_mask is not None:            D_effective = self._apply_tissue_diffusion(D, tissue_mask.to(u.device))        else:            D_effective = D        # --- Diffusion term: ∇·(D(x)∇u) ---        diffusion_term = self.spatial_ops.divergence_of_flux(u, D_effective)        # --- Reaction term ---        if self.pde_model == "fisher_kpp":            reaction_term = rho * u * (1.0 - u)        elif self.pde_model == "gompertz":            reaction_term = rho * u * torch.log(self.K / (u + self.eps))        else:            raise ValueError(f"Unknown PDE model: {self.pde_model}")        # --- PDE Residual ---        rhs = diffusion_term + reaction_term        if du_dt is not None:            # Full PDE residual: R = ∂u/∂t - RHS            residual = du_dt - rhs        else:            # Quasi-steady-state: penalize |RHS|  (assumes ∂u/∂t ≈ 0)            # WEAKER than full PDE — state this limitation explicitly in any paper            residual = rhs        # --- Apply brain mask ---        if mask is not None:            residual = residual * mask            n_voxels = mask.sum().clamp(min=1)        else:            n_voxels = residual.numel()        return (residual ** 2).sum() / n_voxelsclass InitialConditionLoss(nn.Module):    \"\"\"    Initial condition loss: L_IC = ||u(x, t₀) − u₀(x)||².        u₀ can be derived from the earliest available segmentation mask    or from a known initial tumor state.    \"\"\"    def __init__(self, loss_type: str = "mse"):        super().__init__()        self.loss_type = loss_type    def forward(self, u_pred: torch.Tensor, u_initial: torch.Tensor,                mask: torch.Tensor = None) -> torch.Tensor:        \"\"\"        Args:            u_pred: Predicted density at initial time [B, 1, D, H, W].            u_initial: Observed initial density [B, 1, D, H, W].                       Can be derived from segmentation: u₀ = seg > 0.            mask: Optional brain mask.        Returns:            Scalar IC loss.        \"\"\"        diff = u_pred - u_initial        if mask is not None:            diff = diff * mask            n = mask.sum().clamp(min=1)        else:            n = diff.numel()        if self.loss_type == "mse":            return (diff ** 2).sum() / n        elif self.loss_type == "l1":            return diff.abs().sum() / n        elif self.loss_type == "huber":            return F.smooth_l1_loss(u_pred * (mask if mask is not None else 1),                                     u_initial * (mask if mask is not None else 1))        else:            raise ValueError(f"Unknown loss type: {self.loss_type}")class BoundaryConditionLoss(nn.Module):    \"\"\"    No-flux Neumann boundary condition: ∇u · n = 0 at the boundary.        L_BC = (1/N_boundary) Σ |∇u · n|²        The boundary is either the edge of the brain mask or the edge of the volume.    \"\"\"    def __init__(self, dx: float = 1.0, dy: float = 1.0, dz: float = 1.0):        super().__init__()        self.spatial_ops = SpatialGradients3D(dx, dy, dz)    def forward(self, u: torch.Tensor, brain_mask: torch.Tensor = None) -> torch.Tensor:        \"\"\"        Args:            u: Tumor density [B, 1, D, H, W].            brain_mask: Brain mask [B, 1, D, H, W]. Boundary = edge of mask.        Returns:            Scalar BC loss.        \"\"\"        self.spatial_ops = self.spatial_ops.to(u.device)                # Get normal gradient at boundary        boundary_grad = self.spatial_ops.normal_gradient_at_boundary(u, brain_mask)        # Penalize non-zero normal gradient at boundary        n_boundary = (boundary_grad > 0).float().sum().clamp(min=1)        return (boundary_grad ** 2).sum() / n_boundaryclass RegularizationLoss(nn.Module):    \"\"\"    Regularization losses to enforce physical constraints.        L_reg = λ_pos · ||ReLU(−u)||²           (Non-negativity: u ≥ 0)          + λ_bound · ||ReLU(u − 1)||²      (Upper bound: u ≤ 1)            + λ_smooth · ||∇u||²              (Spatial smoothness)          + λ_param · (||D||² + ||ρ||²)      (Parameter regularization)    \"\"\"    def __init__(self, lambda_positivity: float = 10.0,                 lambda_boundedness: float = 10.0,                 lambda_smoothness: float = 0.001,                 lambda_param_reg: float = 0.001,                 dx: float = 1.0, dy: float = 1.0, dz: float = 1.0):        super().__init__()        self.lambda_pos = lambda_positivity        self.lambda_bound = lambda_boundedness        self.lambda_smooth = lambda_smoothness        self.lambda_param = lambda_param_reg        self.spatial_ops = SpatialGradients3D(dx, dy, dz)    def forward(self, u: torch.Tensor, D: torch.Tensor = None,                 rho: torch.Tensor = None, mask: torch.Tensor = None) -> dict:        \"\"\"        Args:            u: Tumor density [B, 1, D, H, W].            D: Diffusion coefficient [B, 1, D, H, W] (optional).            rho: Proliferation rate [B, 1, D, H, W] (optional).            mask: Brain mask [B, 1, D, H, W] (optional).        Returns:            Dictionary with individual regularization terms and total.        \"\"\"        self.spatial_ops = self.spatial_ops.to(u.device)                if mask is not None:            n = mask.sum().clamp(min=1)        else:            n = u.numel()        losses = {}        # --- Non-negativity: u ≥ 0 ---        neg_violation = F.relu(-u)        if mask is not None:            neg_violation = neg_violation * mask        losses["positivity"] = self.lambda_pos * (neg_violation ** 2).sum() / n        # --- Upper bound: u ≤ 1 ---        upper_violation = F.relu(u - 1.0)        if mask is not None:            upper_violation = upper_violation * mask        losses["boundedness"] = self.lambda_bound * (upper_violation ** 2).sum() / n        # --- Spatial smoothness: ||∇u||² ---        grad_mag = self.spatial_ops.gradient_magnitude(u)        if mask is not None:            grad_mag = grad_mag * mask        losses["smoothness"] = self.lambda_smooth * (grad_mag ** 2).sum() / n        # --- Parameter regularization ---        param_loss = torch.tensor(0.0, device=u.device)        if D is not None:            param_loss = param_loss + D.mean()        if rho is not None:            param_loss = param_loss + rho.mean()        losses["param_reg"] = self.lambda_param * param_loss        # Total        losses["total_reg"] = sum(losses.values())        return lossesclass TemporalConsistencyLoss(nn.Module):    \"\"\"    Temporal consistency between predictions at different time points.        Penalizes large jumps in tumor density that violate physical continuity:        L_temporal = ||u(t₂) − [u(t₁) + Δt · f(u(t₁))]||²        where f(u) = ∇·(D∇u) + ρu(1−u) is the PDE right-hand side.    \"\"\"    def __init__(self, dx: float = 1.0, dy: float = 1.0, dz: float = 1.0,                 pde_model: str = "fisher_kpp"):        super().__init__()        self.spatial_ops = SpatialGradients3D(dx, dy, dz)        self.pde_model = pde_model    def forward(self, u_t1: torch.Tensor, u_t2: torch.Tensor,                D: torch.Tensor, rho: torch.Tensor,                delta_t: float, mask: torch.Tensor = None) -> torch.Tensor:        \"\"\"        Args:            u_t1: Tumor density at t₁ [B, 1, D, H, W].            u_t2: Tumor density at t₂ [B, 1, D, H, W].            D: Diffusion coefficient.            rho: Proliferation rate.            delta_t: Time between scans (days).            mask: Brain mask.        Returns:            Scalar temporal consistency loss.        \"\"\"        self.spatial_ops = self.spatial_ops.to(u_t1.device)        # Compute PDE RHS at t₁        diffusion = self.spatial_ops.divergence_of_flux(u_t1, D)                if self.pde_model == "fisher_kpp":            reaction = rho * u_t1 * (1.0 - u_t1)        else:            reaction = rho * u_t1 * torch.log(1.0 / (u_t1 + 1e-7))        rhs = diffusion + reaction        # Forward Euler prediction        u_predicted = u_t1 + delta_t * rhs        u_predicted = torch.clamp(u_predicted, 0.0, 1.0)        # Consistency error        error = u_t2 - u_predicted        if mask is not None:            error = error * mask            n = mask.sum().clamp(min=1)        else:            n = error.numel()        return (error ** 2).sum() / n""")

In [ ]:
with open("losses/data_loss.py", "w") as f:    f.write("""\"\"\"Data-driven loss functions for supervised learning.Segmentation losses, density regression, and seg-to-density conversion.\"\"\"import torchimport torch.nn as nnimport torch.nn.functional as Ffrom monai.losses import DiceLoss, DiceCELoss, FocalLossclass SegmentationLoss(nn.Module):    \"\"\"Segmentation loss: Dice, CE, DiceCE, or Focal.\"\"\"    def __init__(self, loss_type="dice_ce", num_classes=4,                 include_background=False):        super().__init__()        self.loss_type = loss_type        if loss_type == "dice":            self.loss_fn = DiceLoss(to_onehot_y=True, softmax=True,                                     include_background=include_background)        elif loss_type == "ce":            self.loss_fn = nn.CrossEntropyLoss()        elif loss_type == "dice_ce":            self.loss_fn = DiceCELoss(to_onehot_y=True, softmax=True,                                       include_background=include_background)        elif loss_type == "focal":            self.loss_fn = FocalLoss(to_onehot_y=True, gamma=2.0,                                      include_background=include_background)    def forward(self, pred, target):        \"\"\"pred: [B,C,D,H,W] logits, target: [B,1,D,H,W] labels.\"\"\"        if self.loss_type == "ce":            return self.loss_fn(pred, target.squeeze(1).long())        return self.loss_fn(pred, target.long())class DensityRegressionLoss(nn.Module):    \"\"\"MSE/L1/Huber loss for tumor density regression.\"\"\"    def __init__(self, loss_type="mse"):        super().__init__()        self.loss_type = loss_type    def forward(self, pred, target, mask=None):        \"\"\"pred, target: [B,1,D,H,W]. mask: optional brain mask.\"\"\"        diff = pred - target        if mask is not None:            diff = diff * mask            n = mask.sum().clamp(min=1)        else:            n = diff.numel()        if self.loss_type == "mse":            return (diff ** 2).sum() / n        elif self.loss_type == "l1":            return diff.abs().sum() / n        elif self.loss_type == "huber":            return F.smooth_l1_loss(pred, target, reduction="mean")        return (diff ** 2).sum() / ndef seg_to_density(seg_mask, sigma: float = 1.5, smooth: bool = True):    \"\"\"    Convert BraTS segmentation mask to a continuous tumor cell density field.    SCIENTIFIC BASIS (Swanson et al. 2000; Harpold et al. 2007):    ---------------------------------------------------------------    BraTS tumor regions have different expected cell densities:        - ET  (Enhanced Tumor,  class 3): u ≈ 1.0          Contrast-enhancing region: highest intact cell density,          active tumor growth, high vascularization.        - NCR (Necrotic Core,   class 1): u ≈ 0.6          Necrotic/Non-enhancing core: high initial cell mass, but          cells are dead/dying. We assign moderate density (cell debris).        - ED  (Edema,           class 2): u ≈ 0.2          Infiltrative peritumoral edema: sparse tumor cell invasion          into surrounding brain tissue. Cells are present but sparse.        - BG  (Background,      class 0): u = 0.0          No tumor cells.    This mapping is used as the INITIAL CONDITION u(x, t₀) for the PDE.    Optional Gaussian smoothing creates a spatially smooth density field    (avoids sharp discontinuities that cause large PDE residuals at borders).    Args:        seg_mask: [B, 1, D, H, W] integer labels ∈ {0, 1, 2, 3}.                  (After BraTS label remapping: 0=BG, 1=NCR, 2=ED, 3=ET)        sigma: Gaussian smoothing kernel width (mm). Set 0 to disable.        smooth: Whether to apply Gaussian smoothing at region boundaries.    Returns:        [B, 1, D, H, W] float ∈ [0, 1] — tumor cell density.    \"\"\"    seg = seg_mask.float()    device = seg.device    # Biophysically grounded density mapping    density = torch.zeros_like(seg)    density = torch.where(seg == 2, torch.tensor(0.2, device=device), density)  # ED    density = torch.where(seg == 1, torch.tensor(0.6, device=device), density)  # NCR    density = torch.where(seg == 3, torch.tensor(1.0, device=device), density)  # ET    # Optional: Gaussian smoothing at region boundaries to avoid sharp jumps    # This is a SMOOTHING operation, NOT normalization    if smooth and sigma > 0:        ks = max(3, int(4 * sigma + 1))        if ks % 2 == 0:            ks += 1        pad = ks // 2        x = torch.arange(ks, dtype=torch.float32, device=device)        g = torch.exp(-(x - pad) ** 2 / (2.0 * sigma ** 2))        g = g / g.sum()  # Normalize kernel (not the density)        density = F.conv3d(density, g.view(1, 1, -1, 1, 1), padding=(pad, 0, 0))        density = F.conv3d(density, g.view(1, 1, 1, -1, 1), padding=(0, pad, 0))        density = F.conv3d(density, g.view(1, 1, 1, 1, -1), padding=(0, 0, pad))        # Clamp: smoothing can slightly exceed [0,1] at boundaries        density = torch.clamp(density, 0.0, 1.0)    return density""")

In [ ]:
with open("losses/combined_loss.py", "w") as f:    f.write("""\"\"\"Combined Physics-Constrained Loss.L_total = L_data + λ1·L_PDE + λ2·L_IC + λ3·L_BC + λ4·L_regCRITICAL FIX: The PDE loss now uses the FisherKPPSolver to generate aNUMERICALLY INTEGRATED u(t₂) from the current u(t₁), then computesdu/dt = (u_t2 - u_t1) / Δt. This gives a NON-TRIVIAL temporal derivativethat the PDE residual can actually constrain against.The old approach computed du/dt = RHS(u, D, ρ) synthetically, which madethe PDE residual R = du/dt - RHS = 0 by construction — a no-op.Adaptive weight balancing via uncertainty weighting (Kendall et al. 2018).\"\"\"import torchimport torch.nn as nnimport torch.nn.functional as Ffrom .physics_loss import (    PDEResidualLoss, InitialConditionLoss,    BoundaryConditionLoss, RegularizationLoss,    TemporalConsistencyLoss,)from .data_loss import SegmentationLoss, DensityRegressionLoss, seg_to_densityclass AdaptiveWeights(nn.Module):    \"\"\"Learnable log-variance weights (uncertainty-based, Kendall et al. 2018).\"\"\"    def __init__(self, num_losses=5):        super().__init__()        self.log_vars = nn.Parameter(torch.zeros(num_losses))    def forward(self, losses: list) -> torch.Tensor:        total = 0        for i, loss in enumerate(losses):            precision = torch.exp(-self.log_vars[i])            total += precision * loss + self.log_vars[i]        return totalclass HybridTumorLoss(nn.Module):    \"\"\"    Complete loss function combining data-driven and physics losses.    L_total = λ_data · L_data            + λ_pde  · L_PDE            + λ_ic   · L_IC            + λ_bc   · L_BC            + λ_reg  · L_reg            + λ_seg  · L_seg    (pretraining only)    \"\"\"    def __init__(self, config):        super().__init__()        lc = config.loss        pc = config.physics        # Loss weights        self.lambda_data = lc.lambda_data        self.lambda_pde = lc.lambda_pde        self.lambda_ic = lc.lambda_ic        self.lambda_bc = lc.lambda_bc        self.lambda_reg = lc.lambda_reg        self.lambda_seg = lc.lambda_seg        # Data losses        self.seg_loss = SegmentationLoss(lc.seg_loss_type)        self.density_loss = DensityRegressionLoss(lc.data_loss_type)        # Physics losses (pass tissue-aware diffusion parameters)        self.pde_loss = PDEResidualLoss(            pde_model=pc.pde_model,            dx=pc.dx, dy=pc.dy, dz=pc.dz,            carrying_capacity=pc.default_carrying_capacity,            epsilon=pc.epsilon,            D_white_matter=pc.D_white_matter,            D_grey_matter=pc.D_grey_matter,        )        self.ic_loss = InitialConditionLoss()        self.bc_loss = BoundaryConditionLoss(pc.dx, pc.dy, pc.dz)        self.reg_loss = RegularizationLoss(            dx=pc.dx, dy=pc.dy, dz=pc.dz,        )        self.temporal_loss = TemporalConsistencyLoss(            dx=pc.dx, dy=pc.dy, dz=pc.dz,            pde_model=pc.pde_model,        )        # Adaptive weights        self.use_adaptive = lc.use_adaptive_weights        if self.use_adaptive:            self.adaptive = AdaptiveWeights(num_losses=6)        # Default physics params (used when model doesn't predict them)        self.default_D = pc.default_diffusion        self.default_rho = pc.default_proliferation        # Synthetic longitudinal config        self.synthetic_delta_t = pc.synthetic_delta_t        self.use_tissue_aware = pc.tissue_aware_diffusion        # Build a lightweight PDE solver for synthetic du/dt generation        # This is the FisherKPPSolver that numerically integrates the PDE        # forward to produce a NON-TRIVIAL temporal derivative.        self._solver = None  # Lazy init to avoid import issues    def _get_solver(self, device):        \"\"\"Lazy-initialize the FisherKPPSolver on the correct device.\"\"\"        if self._solver is None or self._solver_device != str(device):            from data.synthetic_longitudinal import FisherKPPSolver            self._solver = FisherKPPSolver(                default_D=self.default_D,                default_rho=self.default_rho,                max_dt_step=1.0,                device=str(device),            )            self._solver_device = str(device)        return self._solver    def _generate_synthetic_du_dt(        self,        u: torch.Tensor,        D: torch.Tensor,        rho: torch.Tensor,        brain_mask: torch.Tensor = None,    ) -> torch.Tensor:        \"\"\"        Generate du/dt using NUMERICAL PDE INTEGRATION (not the identity trick).        Strategy:            1. Take current predicted u as u(t₁)            2. Use FisherKPPSolver to integrate the PDE forward by Δt days               using default physics parameters (NOT the network's predictions)            3. Compute du/dt = (u_simulated(t₂) - u(t₁)) / Δt        WHY this works and the old method didn't:            - Old method: du/dt = RHS(u, D_pred, ρ_pred), then loss = |du/dt - RHS|² = 0              This is trivially zero — no training signal.            - New method: du/dt comes from integrating with DEFAULT (D₀, ρ₀),              but the PDE residual tests against the NETWORK's (D_pred, ρ_pred).              The loss is zero ONLY when the network's predictions match the              default parameters AND satisfy the PDE — a real constraint.        The network must learn D(x) and ρ(x) such that the PDE is satisfied        with the observed temporal evolution. This is a genuine physics constraint.        \"\"\"        solver = self._get_solver(u.device)        with torch.no_grad():            # Numerically integrate PDE forward using DEFAULT parameters            # (independent of network predictions — breaks the identity)            u_t2, du_dt = solver.simulate(                u0=u.detach(),                delta_t_days=self.synthetic_delta_t,                D=None,     # Uses solver's default_D (not network's D_pred)                rho=None,   # Uses solver's default_rho (not network's ρ_pred)                brain_mask=brain_mask,                add_noise=False,            )        return du_dt    def _compute_supervised_loss(self, preds, target, loss_fn, is_seg=False, mask=None):        \"\"\"Helper to compute loss with deep supervision.\"\"\"        if not isinstance(preds, list):            preds = [preds]                    total_loss = 0.0        weights = [1.0, 0.5, 0.25]                for i, p in enumerate(preds):            weight = weights[i] if i < len(weights) else 0.1                        # Match spatial dimensions for Deep Supervision            t_down = target            if p.shape[2:] != target.shape[2:]:                t_down = F.interpolate(target.float(), size=p.shape[2:], mode="nearest")                if is_seg:                    t_down = t_down.long()                                m_down = mask            if mask is not None and p.shape[2:] != mask.shape[2:]:                m_down = F.interpolate(mask.float(), size=p.shape[2:], mode="nearest")                        if is_seg:                total_loss += weight * loss_fn(p, t_down)            else:                total_loss += weight * loss_fn(p, t_down, mask=m_down)                        return total_loss    def forward(self, model_output: dict, batch: dict,                phase: str = "joint") -> dict:        \"\"\"        Compute all losses.        Args:            model_output: Dict from HybridTumorNet.forward():                - "tumor_density": [B,1,D,H,W]                - "diffusion": [B,1,D,H,W] (optional)                - "proliferation": [B,1,D,H,W] (optional)                - "segmentation": [B,C,D,H,W] (optional)            batch: Dict from dataloader:                - "image": [B,4,D,H,W]                - "label": [B,1,D,H,W] segmentation                - "du_dt" (optional): real temporal derivative from longitudinal data                - "tissue_mask" (optional): white-matter probability map            phase: "pretrain", "finetune", or "joint"        Returns:            Dict with individual losses and "total_loss".        \"\"\"        losses = {}        u = model_output.get("tumor_density")        seg_pred = model_output.get("segmentation")        label = batch.get("label")        # Create brain mask from input (non-zero voxels = inside brain)        brain_mask = (batch["image"].abs().sum(dim=1, keepdim=True) > 0).float()        # Get physics params        D = model_output.get("diffusion")        rho = model_output.get("proliferation")        if D is None:            D = torch.full_like(u, self.default_D)        if rho is None:            rho = torch.full_like(u, self.default_rho)        # Optional tissue mask for tissue-aware diffusion        tissue_mask = batch.get("tissue_mask", None)        # Base density used for physics losses (always highest resolution)        u_base = u[0] if isinstance(u, list) else u        # === SEGMENTATION LOSS (pretraining) ===        if phase in ("pretrain", "joint") and seg_pred is not None and label is not None:            losses["seg"] = self.lambda_seg * self._compute_supervised_loss(seg_pred, label, self.seg_loss, is_seg=True)        # === DATA FIDELITY LOSS ===        if u is not None and label is not None:            target_density = seg_to_density(label)  # biophysically grounded mapping            losses["data"] = self.lambda_data * self._compute_supervised_loss(                u, target_density, self.density_loss, mask=brain_mask            )        # === PHYSICS LOSSES (finetune / joint) ===        if phase in ("finetune", "joint") and u_base is not None:            # Resolve du/dt:            # Priority 1: real longitudinal du_dt from batch (if available)            # Priority 2: numerically integrated via FisherKPPSolver            du_dt = batch.get("du_dt", None)            if du_dt is not None:                du_dt = du_dt.to(u_base.device)            else:                # Generate synthetic du/dt via numerical PDE integration                # Uses DEFAULT params, NOT network predictions — ensures non-trivial loss                du_dt = self._generate_synthetic_du_dt(u_base, D, rho, brain_mask)            # PDE residual (with optional tissue-aware diffusion)            losses["pde"] = self.lambda_pde * self.pde_loss(                u_base, D, rho,                du_dt=du_dt,                mask=brain_mask,                tissue_mask=tissue_mask,            )            # Initial condition loss            if label is not None:                u_initial = seg_to_density(label)                losses["ic"] = self.lambda_ic * self.ic_loss(                    u_base, u_initial, mask=brain_mask                )            # Boundary condition (no-flux Neumann)            losses["bc"] = self.lambda_bc * self.bc_loss(u_base, brain_mask)            # Regularization (non-negativity, boundedness, smoothness)            reg = self.reg_loss(u_base, D, rho, mask=brain_mask)            losses["reg"] = self.lambda_reg * reg["total_reg"]            # Temporal consistency (if real longitudinal data available)            if "image_t2" in batch and "density_t2" in batch:                u_t2 = batch["density_t2"].to(u_base.device)                delta_t = batch.get("delta_t", 90.0)                if isinstance(delta_t, torch.Tensor):                    delta_t = delta_t.mean().item()                losses["temporal"] = 0.05 * self.temporal_loss(                    u_base, u_t2, D, rho, delta_t, mask=brain_mask                )        # === TOTAL ===        if self.use_adaptive and len(losses) > 1:            loss_list = list(losses.values())            losses["total_loss"] = self.adaptive(loss_list)        else:            losses["total_loss"] = sum(losses.values())        return losses""")

In [ ]:
with open("config.py", "w") as f:    f.write("""\"\"\"Configuration for Physics-Constrained ResNet Brain Tumor Model.All hyperparameters, paths, and settings centralized here.SCALE GUIDE (VRAM requirements):    resnet10  + spatial=(64,64,64)  + batch=1 → ~4–6 GB   (any gaming GPU)    resnet18  + spatial=(64,64,64)  + batch=2 → ~8–10 GB  (RTX 3070+)    resnet50  + spatial=(128,128,128)+ batch=2 → ~40–80 GB (A100 only)For LOCAL DEVELOPMENT use the defaults (resnet10, 64³).Change backbone to resnet50 and spatial_size to (128,128,128) only forfinal paper experiments on cloud GPU (Colab A100, Kaggle P100).\"\"\"import osfrom dataclasses import dataclass, fieldfrom typing import List, Tuple, Optional@dataclassclass DataConfig:    \"\"\"Dataset and preprocessing configuration.\"\"\"    # --- Paths ---    data_root: str = "./data/Task01_BrainTumour"    train_csv: str = "./data/train.csv"    val_csv: str = "./data/val.csv"    test_csv: str = "./data/test.csv"    # --- MRI Modalities ---    modalities: List[str] = field(default_factory=lambda: ["t1", "t1ce", "t2", "flair"])    num_channels: int = 4  # Number of input MRI modalities    # --- Volume dimensions ---    # DEFAULT: 64³ for local GPU training (4–6 GB VRAM)    # For paper results, change to (128, 128, 128) on cloud GPU    spatial_size: Tuple[int, int, int] = (64, 64, 64)  # Crop/resize target    voxel_spacing: Tuple[float, float, float] = (1.0, 1.0, 1.0)  # mm    # --- Segmentation labels (BraTS) ---    num_seg_classes: int = 4  # Background, NCR/NET, ED, ET    # BraTS label mapping: 0=background, 1=NCR/NET, 2=ED, 4=ET    label_mapping: dict = field(default_factory=lambda: {0: 0, 1: 1, 2: 2, 4: 3})    # --- Augmentation ---    rand_flip_prob: float = 0.5    rand_rotate_prob: float = 0.3    rand_scale_prob: float = 0.3    rand_noise_std: float = 0.1    intensity_shift_range: float = 0.1    intensity_scale_range: float = 0.1@dataclassclass ModelConfig:    \"\"\"Model architecture configuration.\"\"\"    # --- Backbone ---    # DEFAULT: resnet10 for local GPU training (~1.7M params)    # Change to resnet50 for paper experiments on cloud GPU (~46M params)    backbone: str = "resnet10"  # resnet10, resnet18, resnet34, resnet50, resnet101, resnet152, resnet200    pretrained: bool = True    pretrained_weights_path: Optional[str] = None  # Path to MedicalNet weights    # --- Feature dimensions ---    # MUST match backbone: resnet10/18/34 → 512, resnet50/101/152/200 → 2048    feature_dim: int = 512   # resnet10 output features    hidden_dim: int = 256    # --- Tumor Density Decoder ---    # Scaled down for resnet10; use [256,128,64,32,16] for resnet50    decoder_channels: List[int] = field(default_factory=lambda: [128, 64, 32, 16, 8])    decoder_use_skip: bool = True  # Use skip connections from encoder    # --- Physics Parameter Head ---    predict_diffusion: bool = True  # Predict D(x) spatially    predict_proliferation: bool = True  # Predict ρ    diffusion_range: Tuple[float, float] = (0.0001, 0.5)  # mm²/day    proliferation_range: Tuple[float, float] = (0.001, 0.5)  # 1/day    carrying_capacity: float = 1.0  # K in logistic growth    # --- Segmentation head (for pretraining) ---    use_seg_head: bool = True    num_seg_classes: int = 4    # --- Dropout ---    dropout_rate: float = 0.3@dataclassclass PhysicsConfig:    \"\"\"    Physics-constrained loss configuration.    PDE: Fisher-KPP reaction-diffusion equation (Swanson et al. 2000)        ∂u/∂t = ∇·(D(x)∇u) + ρ·u·(1 − u)    Biophysically grounded default parameters:        D_wm  ≈ 0.05–0.10 mm²/day (white matter, Harpold et al. 2007)        D_gm  ≈ 0.01–0.02 mm²/day (grey matter, ≈5x lower)        ρ     ≈ 0.012–0.025 /day  (Swanson et al. 2000)    \"\"\"    # --- PDE Model ---    pde_model: str = "fisher_kpp"  # "fisher_kpp" or "gompertz"    # --- Default physics parameters (used if not predicted by network) ---    # Source: Swanson et al. (2000), Harpold et al. (2007)    default_diffusion: float = 0.05   # mm²/day — conservative/stable default    default_proliferation: float = 0.02  # 1/day    default_carrying_capacity: float = 1.0    # --- Spatial resolution for finite differences ---    dx: float = 1.0  # voxel spacing in mm (x)    dy: float = 1.0  # voxel spacing in mm (y)    dz: float = 1.0  # voxel spacing in mm (z)    # --- Temporal ---    # Default Δt for synthetic longitudinal pairs (days)    dt: float = 90.0  # ~3 months — typical glioma re-scan interval    synthetic_delta_t: float = 90.0  # days for PDE simulation in training    # --- Tissue-aware diffusion (Harpold et al. 2007) ---    # White matter is ~5-10x more diffusive than grey matter for glioma    tissue_aware_diffusion: bool = True    D_white_matter: float = 0.08   # mm²/day (white matter diffusion)    D_grey_matter: float = 0.016   # mm²/day (grey matter diffusion)    # Ratio: D_wm / D_gm ≈ 5 (biophysically validated)    diffusion_ratio_wm_gm: float = 5.0    # --- Boundary conditions ---    boundary_type: str = "neumann"  # "neumann" (no-flux) — standard for glioma    # --- Constraints ---    enforce_non_negativity: bool = True    enforce_upper_bound: bool = True    upper_bound_value: float = 1.0    epsilon: float = 1e-7  # Small constant to avoid log(0) in Gompertz@dataclassclass LossConfig:    \"\"\"Loss function weights and configuration.\"\"\"    # --- Loss weights ---    lambda_data: float = 1.0       # Data fidelity loss    lambda_pde: float = 0.1        # PDE residual loss    lambda_ic: float = 0.05        # Initial condition loss    lambda_bc: float = 0.01        # Boundary condition loss    lambda_reg: float = 0.01       # Regularization loss    lambda_seg: float = 1.0        # Segmentation loss (pretraining)    # --- Adaptive weighting ---    use_adaptive_weights: bool = True    adaptive_method: str = "grad_norm"  # "grad_norm", "uncertainty", "manual"    # --- Data loss type ---    data_loss_type: str = "mse"  # "mse", "l1", "huber"    seg_loss_type: str = "dice_ce"  # "dice", "ce", "dice_ce", "focal"    # --- Regularization ---    smoothness_weight: float = 0.001    l2_weight_decay: float = 1e-5@dataclassclass TrainConfig:    \"\"\"Training configuration.\"\"\"    # --- Phase ---    phase: str = "pretrain"  # "pretrain" (seg only), "finetune" (physics), "joint"    # --- Optimization ---    # batch_size=1 for local (resnet10+64³); increase to 2 on cloud GPU (resnet50+128³)    batch_size: int = 1    accumulation_steps: int = 4  # Simulates larger batch size (1 * 4 = effect of batch 4)    num_epochs_pretrain: int = 1    num_epochs_finetune: int = 1    learning_rate: float = 1e-4    lr_backbone: float = 1e-5  # Lower LR for pretrained backbone    weight_decay: float = 1e-5    optimizer: str = "adamw"  # "adam", "adamw", "sgd"    # --- Scheduler ---    scheduler: str = "cosine"  # "cosine", "step", "plateau", "warmup_cosine", "cosine_warm_restarts"    warmup_epochs: int = 10    min_lr: float = 1e-7    # --- Mixed precision ---    use_amp: bool = True    grad_clip_norm: float = 1.0    # --- Checkpointing ---    checkpoint_dir: str = "./checkpoints"    save_every: int = 10    resume_from: Optional[str] = None    # --- Logging ---    log_dir: str = "./logs"    use_wandb: bool = False    wandb_project: str = "brain-tumor-pinn"    # --- Validation ---    val_every: int = 5    early_stopping_patience: int = 30    # --- Device ---    device: str = "cuda"    num_workers: int = 4    pin_memory: bool = True    # --- Reproducibility ---    seed: int = 42@dataclassclass Config:    \"\"\"Master configuration.\"\"\"    data: DataConfig = field(default_factory=DataConfig)    model: ModelConfig = field(default_factory=ModelConfig)    physics: PhysicsConfig = field(default_factory=PhysicsConfig)    loss: LossConfig = field(default_factory=LossConfig)    train: TrainConfig = field(default_factory=TrainConfig)    def __post_init__(self):        \"\"\"Create necessary directories.\"\"\"        os.makedirs(self.train.checkpoint_dir, exist_ok=True)        os.makedirs(self.train.log_dir, exist_ok=True)def get_config(**overrides) -> Config:    \"\"\"Get configuration with optional overrides.\"\"\"    config = Config()    for key, value in overrides.items():        parts = key.split(".")        obj = config        for part in parts[:-1]:            obj = getattr(obj, part)        setattr(obj, parts[-1], value)    return config""")

In [ ]:
with open("train.py", "w") as f:    f.write("""\"\"\"Training script for Hybrid MONAI ResNet + Physics-Informed Tumor Model.Two-phase training:    Phase 1 (Pretrain): Train segmentation head on BraTS data    Phase 2 (Finetune): Fine-tune with physics-informed losses                        + synthetic longitudinal du/dt from FisherKPPSolverUsage:    python train.py --data_root ./data/Task01_BrainTumour --phase pretrain    python train.py --data_root ./data/Task01_BrainTumour --phase finetune --resume checkpoints/pretrain_best.pth    python train.py --data_root ./data/Task01_BrainTumour --phase joint\"\"\"import osimport sysimport timeimport argparseimport jsonimport numpy as npimport torchimport torch.nn as nnfrom torch.amp import GradScaler, autocastfrom torch.utils.tensorboard import SummaryWriterfrom tqdm import tqdmfrom config import Config, get_configfrom models.hybrid_model import HybridTumorNetfrom losses.combined_loss import HybridTumorLossfrom losses.data_loss import seg_to_densityfrom data.dataset import get_train_val_dataloadersfrom utils.metrics import compute_dice, compute_physics_residualfrom utils.spatial_ops import SpatialGradients3Dfrom utils.ema import EMAModeldef set_seed(seed):    torch.manual_seed(seed)    torch.cuda.manual_seed_all(seed)    np.random.seed(seed)    torch.backends.cudnn.deterministic = Truedef build_model(config):    \"\"\"Create the HybridTumorNet model.\"\"\"    mc = config.model    model = HybridTumorNet(        resnet_variant=mc.backbone,        in_channels=config.data.num_channels,        num_seg_classes=mc.num_seg_classes,        pretrained_path=mc.pretrained_weights_path,        freeze_encoder_layers=0,        decoder_channels=tuple(mc.decoder_channels),        dropout=mc.dropout_rate,        use_seg_head=mc.use_seg_head,        predict_physics_params=mc.predict_diffusion,        diffusion_range=mc.diffusion_range,        proliferation_range=mc.proliferation_range,    )    params = model.count_parameters()    print(f"Model parameters: {params['total']:,} total, "          f"{params['trainable']:,} trainable")    return modeldef build_optimizer(model, config):    \"\"\"Create optimizer with differential learning rates.\"\"\"    tc = config.train    param_groups = model.get_parameter_groups(        lr_backbone=tc.lr_backbone,        lr_head=tc.learning_rate,    )    if tc.optimizer == "adamw":        return torch.optim.AdamW(param_groups, weight_decay=tc.weight_decay)    elif tc.optimizer == "adam":        return torch.optim.Adam(param_groups, weight_decay=tc.weight_decay)    elif tc.optimizer == "sgd":        return torch.optim.SGD(param_groups, momentum=0.9,                                weight_decay=tc.weight_decay)    raise ValueError(f"Unknown optimizer: {tc.optimizer}")def build_scheduler(optimizer, config, num_epochs):    \"\"\"Create learning rate scheduler.\"\"\"    tc = config.train    if tc.scheduler == "cosine":        return torch.optim.lr_scheduler.CosineAnnealingLR(            optimizer, T_max=num_epochs, eta_min=tc.min_lr        )    elif tc.scheduler == "plateau":        return torch.optim.lr_scheduler.ReduceLROnPlateau(            optimizer, mode="min", factor=0.5, patience=10        )    elif tc.scheduler == "warmup_cosine":        def lr_lambda(epoch):            if epoch < tc.warmup_epochs:                return (epoch + 1) / tc.warmup_epochs            progress = (epoch - tc.warmup_epochs) / (num_epochs - tc.warmup_epochs)            return max(tc.min_lr / tc.learning_rate,                       0.5 * (1 + np.cos(np.pi * progress)))        return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)    elif tc.scheduler == "cosine_warm_restarts":        return torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(            optimizer, T_0=50, T_mult=2, eta_min=tc.min_lr        )    return Nonedef generate_synthetic_du_dt_for_batch(batch, config, device):    \"\"\"    Generate synthetic temporal derivatives for physics-informed training.    Uses FisherKPPSolver to numerically integrate the Fisher-KPP PDE    from the segmentation-derived density u(t₁), producing:        - du/dt = (u_simulated(t₂) - u(t₁)) / Δt    This provides the temporal derivative needed for the PDE residual loss    when real longitudinal data is not available (cross-sectional datasets).    Args:        batch: Training batch dict with "label" key.        config: Config object.        device: torch device.    Returns:        Updated batch dict with "du_dt" key added.    \"\"\"    from data.synthetic_longitudinal import FisherKPPSolver    label = batch.get("label")    if label is None:        return batch    pc = config.physics    # Convert segmentation to density field (initial condition)    u_t1 = seg_to_density(label, sigma=1.0, smooth=True).to(device)    # Create brain mask from input    brain_mask = (batch["image"].abs().sum(dim=1, keepdim=True) > 0).float().to(device)    # Initialize solver with default biophysical parameters    solver = FisherKPPSolver(        default_D=pc.default_diffusion,        default_rho=pc.default_proliferation,        max_dt_step=1.0,        device=str(device),    )    # Simulate tumor growth forward by synthetic_delta_t days    u_t2, du_dt = solver.simulate(        u0=u_t1,        delta_t_days=pc.synthetic_delta_t,        brain_mask=brain_mask,        add_noise=True,        noise_std=0.005,    )    batch["du_dt"] = du_dt    batch["density_t2"] = u_t2    batch["delta_t"] = torch.tensor(pc.synthetic_delta_t)    return batchdef train_one_epoch(model, loader, criterion, optimizer, scaler,                     device, phase, epoch, config, writer=None, ema_model=None):    \"\"\"Train for one epoch.\"\"\"    model.train()    epoch_losses = {}    num_batches = 0    accumulation_steps = getattr(config.train, 'accumulation_steps', 1)    pbar = tqdm(loader, desc=f"Epoch {epoch} [{phase}]")    for batch_idx, batch in enumerate(pbar):        # Move to device        images = batch["image"].to(device)        labels = batch["label"].to(device)        batch_gpu = {"image": images, "label": labels}        # For finetune/joint phases: generate synthetic temporal derivatives        # This wires the FisherKPPSolver into the training loop        if phase in ("finetune", "joint"):            batch_gpu = generate_synthetic_du_dt_for_batch(batch_gpu, config, device)        optimizer.zero_grad()        # Forward pass        with autocast(device_type=device.type, enabled=scaler is not None):            output = model(images)            loss_dict = criterion(output, batch_gpu, phase=phase)            total_loss = loss_dict["total_loss"] / accumulation_steps        # Backward pass        if scaler is not None:            scaler.scale(total_loss).backward()            if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(loader):                scaler.unscale_(optimizer)                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)                scaler.step(optimizer)                scaler.update()                optimizer.zero_grad()                if ema_model is not None:                    ema_model.update(model)        else:            total_loss.backward()            if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(loader):                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)                optimizer.step()                optimizer.zero_grad()                if ema_model is not None:                    ema_model.update(model)        # Track losses        for k, v in loss_dict.items():            if isinstance(v, torch.Tensor):                v = v.item()            # If we divided total_loss by accumulation_steps above, we should             # log the actual non-divided value for clarity, but they are from loss_dict so it's fine.            epoch_losses[k] = epoch_losses.get(k, 0) + v        num_batches += 1        # Progress bar        pbar.set_postfix({            "loss": f"{total_loss.item():.4f}",            "lr": f"{optimizer.param_groups[0]['lr']:.2e}",        })    # Average losses    for k in epoch_losses:        epoch_losses[k] /= max(num_batches, 1)    # Tensorboard    if writer:        for k, v in epoch_losses.items():            writer.add_scalar(f"train/{k}", v, epoch)        writer.add_scalar("train/lr", optimizer.param_groups[0]["lr"], epoch)    return epoch_losses@torch.no_grad()def validate(model, loader, criterion, device, phase, epoch, config, writer=None):    \"\"\"Validate the model.\"\"\"    model.eval()    epoch_losses = {}    all_dice = []    num_batches = 0    for batch in tqdm(loader, desc=f"Val {epoch}"):        images = batch["image"].to(device)        labels = batch["label"].to(device)        batch_gpu = {"image": images, "label": labels}        # Generate synthetic du/dt for validation too (consistent evaluation)        if phase in ("finetune", "joint"):            batch_gpu = generate_synthetic_du_dt_for_batch(batch_gpu, config, device)        output = model(images)        loss_dict = criterion(output, batch_gpu, phase=phase)        for k, v in loss_dict.items():            if isinstance(v, torch.Tensor):                v = v.item()            epoch_losses[k] = epoch_losses.get(k, 0) + v        # Dice scores        if "segmentation" in output:            dice = compute_dice(output["segmentation"], labels)            all_dice.append(dice)        num_batches += 1    for k in epoch_losses:        epoch_losses[k] /= max(num_batches, 1)    # Average dice    if all_dice:        avg_dice = {}        for k in all_dice[0]:            avg_dice[k] = np.mean([d[k] for d in all_dice])        epoch_losses.update(avg_dice)    if writer:        for k, v in epoch_losses.items():            writer.add_scalar(f"val/{k}", v, epoch)    return epoch_lossesdef save_checkpoint(model, optimizer, scheduler, epoch, losses, path, ema_model=None):    \"\"\"Save model checkpoint.\"\"\"    checkpoint_state = {        "epoch": epoch,        "model_state_dict": model.state_dict(),        "optimizer_state_dict": optimizer.state_dict(),        "scheduler_state_dict": scheduler.state_dict() if scheduler else None,        "losses": losses,    }    if ema_model is not None:        checkpoint_state["ema_model_state_dict"] = ema_model.state_dict()        torch.save(checkpoint_state, path)    print(f"  Saved checkpoint: {path}")def load_checkpoint(model, optimizer, scheduler, path, device, ema_model=None):    \"\"\"Load model checkpoint.\"\"\"    ckpt = torch.load(path, map_location=device)    model.load_state_dict(ckpt["model_state_dict"], strict=False)    if optimizer and "optimizer_state_dict" in ckpt:        try:            optimizer.load_state_dict(ckpt["optimizer_state_dict"])        except Exception:            print("  Warning: Could not load optimizer state")    if scheduler and ckpt.get("scheduler_state_dict"):        try:            scheduler.load_state_dict(ckpt["scheduler_state_dict"])        except Exception:            pass    if ema_model and "ema_model_state_dict" in ckpt:        try:            ema_model.load_state_dict(ckpt["ema_model_state_dict"], strict=False)            print("  Loaded EMA model state")        except Exception:            print("  Warning: Could not load EMA model state")                print(f"  Loaded checkpoint from epoch {ckpt.get('epoch', '?')}")    return ckpt.get("epoch", 0)def train(config: Config):    \"\"\"Main training function.\"\"\"    tc = config.train    set_seed(tc.seed)    device = torch.device(tc.device if torch.cuda.is_available() else "cpu")    print(f"Device: {device}")    # Data    print("Loading data...")    train_loader, val_loader = get_train_val_dataloaders(        data_root=config.data.data_root,        batch_size=tc.batch_size,        spatial_size=config.data.spatial_size,        num_workers=0,  # CRITICAL FOR WINDOWS: Avoids Multiprocessing OOM / Deadlocks        cache_rate=0.0, # Do not cache into RAM on machines with low memory    )    # Model    print("Building model...")    model = build_model(config).to(device)    model.set_phase(tc.phase)        ema_model = EMAModel(model, device=device)    # Loss    criterion = HybridTumorLoss(config).to(device)    # Optimizer & scheduler    num_epochs = (tc.num_epochs_pretrain if tc.phase == "pretrain"                  else tc.num_epochs_finetune)    optimizer = build_optimizer(model, config)    scheduler = build_scheduler(optimizer, config, num_epochs)    # AMP scaler    scaler = GradScaler(device.type) if tc.use_amp and device.type == "cuda" else None    # Resume    start_epoch = 0    if tc.resume_from and os.path.exists(tc.resume_from):        start_epoch = load_checkpoint(model, optimizer, scheduler,                                       tc.resume_from, device, ema_model)    # Tensorboard    writer = SummaryWriter(log_dir=os.path.join(tc.log_dir, tc.phase))    # Training loop    best_val_loss = float("inf")    patience_counter = 0    print(f"\\n{'='*60}")    print(f"Training Phase: {tc.phase.upper()}")    print(f"Epochs: {start_epoch} -> {num_epochs}")    print(f"Batch size: {tc.batch_size}")    if tc.phase in ("finetune", "joint"):        print(f"Synthetic Δt: {config.physics.synthetic_delta_t} days")        print(f"PDE model: {config.physics.pde_model}")    print(f"{'='*60}\\n")    # JSON metrics log    metrics_log_path = os.path.join(tc.log_dir, f"{tc.phase}_metrics.jsonl")    for epoch in range(start_epoch, num_epochs):        t0 = time.time()        # Disable automatic zero_grad in loop start since we handle it in accumulation        # Train        train_losses = train_one_epoch(            model, train_loader, criterion, optimizer, scaler,            device, tc.phase, epoch, config, writer, ema_model        )        # Validate (using EMA model if available)        val_losses = {}        if (epoch + 1) % tc.val_every == 0:            val_model = ema_model if ema_model is not None else model            val_losses = validate(                val_model, val_loader, criterion, device,                tc.phase, epoch, config, writer            )        # Scheduler step        if scheduler:            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):                scheduler.step(val_losses.get("total_loss", train_losses["total_loss"]))            else:                scheduler.step()        # Logging        elapsed = time.time() - t0                # Log GPU memory        gpu_mem = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0                print(f"Epoch {epoch:3d} | "              f"Train Loss: {train_losses['total_loss']:.4f} | "              f"Val Loss: {val_losses.get('total_loss', 'N/A')} | "              f"Time: {elapsed:.1f}s | GPU Mem: {gpu_mem:.0f}MB")        if val_losses:            for k, v in val_losses.items():                if "dice" in k:                    print(f"  {k}: {v:.4f}")                            # Log metrics to JSONL        epoch_metrics = {            "epoch": epoch,            "phase": tc.phase,            "train": train_losses,            "val": val_losses,            "time_sec": elapsed,            "gpu_mem_mb": gpu_mem,            "lr": optimizer.param_groups[0]["lr"]        }        with open(metrics_log_path, "a") as f:            f.write(json.dumps(epoch_metrics) + "\\n")        # Save best        val_loss = val_losses.get("total_loss", train_losses["total_loss"])        if val_loss < best_val_loss:            best_val_loss = val_loss            patience_counter = 0            save_checkpoint(                model, optimizer, scheduler, epoch, val_losses,                os.path.join(tc.checkpoint_dir, f"{tc.phase}_best.pth"),                ema_model=ema_model            )        else:            patience_counter += 1        # Periodic save        if (epoch + 1) % tc.save_every == 0:            save_checkpoint(                model, optimizer, scheduler, epoch, val_losses,                os.path.join(tc.checkpoint_dir, f"{tc.phase}_epoch{epoch}.pth"),                ema_model=ema_model            )        # Early stopping        if patience_counter >= tc.early_stopping_patience:            print(f"Early stopping at epoch {epoch}")            break    writer.close()    print(f"\\nTraining complete. Best val loss: {best_val_loss:.4f}")    return modeldef main():    parser = argparse.ArgumentParser(description="Train Hybrid Tumor Model")    parser.add_argument("--data_root", type=str, default="./data/Task01_BrainTumour")    parser.add_argument("--phase", type=str, default="pretrain",                        choices=["pretrain", "finetune", "joint"])    parser.add_argument("--backbone", type=str, default="resnet10")    parser.add_argument("--batch_size", type=int, default=1)    parser.add_argument("--epochs", type=int, default=None)    parser.add_argument("--lr", type=float, default=1e-4)    parser.add_argument("--resume", type=str, default=None)    parser.add_argument("--device", type=str, default="cuda")    parser.add_argument("--pde_model", type=str, default="fisher_kpp",                        choices=["fisher_kpp", "gompertz"])    args = parser.parse_args()    config = get_config()    config.data.data_root = args.data_root    config.train.phase = args.phase    config.model.backbone = args.backbone    config.train.batch_size = args.batch_size    config.train.learning_rate = args.lr    config.train.device = args.device    config.physics.pde_model = args.pde_model    if args.resume:        config.train.resume_from = args.resume    if args.epochs:        if args.phase == "pretrain":            config.train.num_epochs_pretrain = args.epochs        else:            config.train.num_epochs_finetune = args.epochs    train(config)if __name__ == "__main__":    main()""")

In [ ]:
with open("predict.py", "w") as f:    f.write("""\"\"\"Inference & Prediction Script for Hybrid Tumor Model.Given an MRI scan, this script:    1. Loads the trained model    2. Predicts tumor density u(x,t)    3. Predicts physics parameters D(x), ρ(x)    4. Simulates tumor growth forward in time    5. Generates segmentation    6. Saves visualizations and NIfTI outputsUsage:    python predict.py --input_dir ./patient_scan --checkpoint checkpoints/finetune_best.pth    python predict.py --input_dir ./patient_scan --checkpoint checkpoints/finetune_best.pth --simulate_days 180\"\"\"import osimport argparseimport numpy as npimport torchimport torch.nn as nnimport nibabel as nibimport matplotlibmatplotlib.use("Agg")import matplotlib.pyplot as pltfrom matplotlib.colors import LinearSegmentedColormapfrom monai.inferers import sliding_window_inferencefrom config import get_configfrom models.hybrid_model import HybridTumorNetfrom data.preprocessing import get_inference_transformsfrom utils.spatial_ops import SpatialGradients3Ddef load_model(checkpoint_path, config, device):    \"\"\"Load trained model from checkpoint.\"\"\"    model = HybridTumorNet(        resnet_variant=config.model.backbone,        in_channels=config.data.num_channels,        num_seg_classes=config.model.num_seg_classes,        use_seg_head=config.model.use_seg_head,        predict_physics_params=config.model.predict_diffusion,        diffusion_range=config.model.diffusion_range,        proliferation_range=config.model.proliferation_range,    )    if checkpoint_path == "random":        print("Using randomly initialized weights for demonstration purposes.")        model = model.to(device)        model.eval()        return model    if os.path.exists(checkpoint_path):        ckpt = torch.load(checkpoint_path, map_location=device)        if "ema_model_state_dict" in ckpt:            model.load_state_dict(ckpt["ema_model_state_dict"], strict=False)            print("Loaded EMA weights from checkpoint.")        elif "model_state_dict" in ckpt:            model.load_state_dict(ckpt["model_state_dict"], strict=False)        else:            model.load_state_dict(ckpt, strict=False)        epoch_str = ckpt.get('epoch', '?')    else:        print(f"Warning: Checkpoint {checkpoint_path} not found. Using random weights.")        epoch_str = 'unknown'            model = model.to(device)    model.eval()    print(f"Loaded model from {checkpoint_path} (epoch {epoch_str})")    return modeldef load_patient_scan(input_dir, spatial_size=(128, 128, 128)):    \"\"\"Load and preprocess a patient's MRI scan.\"\"\"    import glob    # Check if input is a direct 4-channel file (like MSD format)    if os.path.isfile(input_dir) and input_dir.endswith(".nii.gz"):        print(f"Loading single multi-channel scan: {os.path.basename(input_dir)}")        files = input_dir        ref_img = nib.load(input_dir)        affine = ref_img.affine    else:        # BraTS format: Find modality files        modalities = ["t1", "t1ce", "t2", "flair"]        files = []        for mod in modalities:            matches = glob.glob(os.path.join(input_dir, f"*_{mod}.nii.gz"))            if not matches:                matches = glob.glob(os.path.join(input_dir, f"*{mod}*.nii.gz"))            if matches:                files.append(matches[0])            else:                raise FileNotFoundError(f"Could not find {mod} modality in {input_dir}")        print(f"Found modalities: {[os.path.basename(f) for f in files]}")        ref_img = nib.load(files[0])        affine = ref_img.affine    # Load with MONAI transforms    transforms = get_inference_transforms(spatial_size)    data = {"image": files}    data = transforms(data)    return data["image"].unsqueeze(0), affine  # Add batch dim@torch.no_grad()def predict(model, image, device, simulate_days=0, dt=1.0, spatial_size=(128,128,128), mc_samples=0):    \"\"\"    Run full prediction pipeline.    Args:        model: Trained HybridTumorNet.        image: Preprocessed MRI [1, 4, D, H, W].        device: torch device.        simulate_days: If > 0, simulate tumor growth forward.        dt: Time step for simulation (days).        spatial_size: Spatial size for sliding window inference.        mc_samples: Number of Monte Carlo samples (0 for standard deterministic prediction).    Returns:        Dictionary with all predictions.    \"\"\"    image = image.to(device)        use_mc_dropout = mc_samples > 1    num_samples = mc_samples if use_mc_dropout else 1    if use_mc_dropout:        model.train()  # Enable dropout    else:        model.eval()    def forward_wrapper(x):        out = model(x, return_features=False)        td = out["tumor_density"]        if isinstance(td, list): td = td[0]                diff = out.get("diffusion", torch.zeros_like(td))        prolif = out.get("proliferation", torch.zeros_like(td))                seg = out.get("segmentation", torch.zeros_like(td).repeat(1, 4, 1, 1, 1))                return torch.cat([td, diff, prolif, seg], dim=1)    all_outputs = []    print(f"Running inference (Sliding Window, ROI: {spatial_size})...")        for idx in range(num_samples):        if use_mc_dropout:            print(f"  MC Dropout Sample {idx+1}/{num_samples}...")                full_out = sliding_window_inference(            inputs=image,            roi_size=spatial_size,            sw_batch_size=1,            predictor=forward_wrapper,            overlap=0.5        )        all_outputs.append(full_out)    all_outputs = torch.stack(all_outputs, dim=0) # [samples, B, C, D, H, W]    mean_out = all_outputs.mean(dim=0)    std_out = all_outputs.std(dim=0) if use_mc_dropout else None    results = {        "tumor_density": mean_out[0, 0].cpu().numpy(),        "diffusion": mean_out[0, 1].cpu().numpy(),        "proliferation": mean_out[0, 2].cpu().numpy(),    }        if use_mc_dropout:        results["uncertainty"] = std_out[0, 0].cpu().numpy()    # Segmentation    seg_logits = mean_out[:, 3:]    if seg_logits.shape[1] > 0 and seg_logits.sum() != 0:        seg = torch.argmax(seg_logits, dim=1)        results["segmentation"] = seg.cpu().numpy()[0]    # Tumor growth simulation    if simulate_days > 0:        print(f"Simulating tumor growth for {simulate_days} days...")        num_steps = int(simulate_days / dt)        trajectory = model.predict_tumor_growth(            image, num_steps=num_steps, dt=dt        )        results["growth_trajectory"] = [            t.cpu().numpy()[0, 0] for t in trajectory        ]        print(f"  Generated {len(trajectory)} time steps")    # Compute tumor statistics    u = results["tumor_density"]    results["stats"] = {        "tumor_volume_voxels": int((u > 0.5).sum()),        "tumor_volume_ml": float((u > 0.5).sum() * 0.001),  # assuming 1mm³ voxels        "max_density": float(u.max()),        "mean_density_in_tumor": float(u[u > 0.1].mean()) if (u > 0.1).any() else 0,        "mean_diffusion": float(results["diffusion"].mean()) if results["diffusion"].ndim > 1 else 0,        "mean_proliferation": float(results["proliferation"].mean()) if results["proliferation"].ndim > 1 else 0,    }    return resultsdef save_nifti(data, affine, path):    \"\"\"Save a 3D numpy array as NIfTI file.\"\"\"    img = nib.Nifti1Image(data.astype(np.float32), affine)    nib.save(img, path)    print(f"  Saved: {path}")def visualize_results(results, output_dir, num_slices=5):    \"\"\"Generate visualization plots.\"\"\"    os.makedirs(output_dir, exist_ok=True)    u = results["tumor_density"]    D, H, W = u.shape    # Custom colormap: dark blue → cyan → yellow → red    colors = ["#0d0887", "#46039f", "#7201a8", "#9c179e",              "#bd3786", "#d8576b", "#ed7953", "#fb9f3a", "#fdca26", "#f0f921"]    tumor_cmap = LinearSegmentedColormap.from_list("tumor", colors)    # --- 1. Tumor Density Slices ---    fig, axes = plt.subplots(1, num_slices, figsize=(4 * num_slices, 4))    slice_indices = np.linspace(D * 0.3, D * 0.7, num_slices, dtype=int)    for i, si in enumerate(slice_indices):        ax = axes[i] if num_slices > 1 else axes        im = ax.imshow(u[si], cmap=tumor_cmap, vmin=0, vmax=1)        ax.set_title(f"Slice {si}", fontsize=12)        ax.axis("off")    plt.colorbar(im, ax=axes, shrink=0.8, label="Tumor Density")    plt.suptitle("Predicted Tumor Density u(x,t)", fontsize=14, fontweight="bold")    plt.tight_layout()    plt.savefig(os.path.join(output_dir, "tumor_density.png"), dpi=150, bbox_inches="tight")    plt.close()    # --- 2. Segmentation (if available) ---    if "segmentation" in results:        seg = results["segmentation"]        fig, axes = plt.subplots(1, num_slices, figsize=(4 * num_slices, 4))        seg_cmap = plt.cm.get_cmap("Set1", 4)        for i, si in enumerate(slice_indices):            ax = axes[i] if num_slices > 1 else axes            ax.imshow(seg[si], cmap=seg_cmap, vmin=0, vmax=3)            ax.set_title(f"Slice {si}", fontsize=12)            ax.axis("off")        plt.suptitle("Segmentation (0:BG, 1:NCR, 2:ED, 3:ET)", fontsize=14)        plt.tight_layout()        plt.savefig(os.path.join(output_dir, "segmentation.png"), dpi=150, bbox_inches="tight")        plt.close()    # --- 3. Physics Parameters ---    if results["diffusion"].ndim > 1:        fig, axes = plt.subplots(2, num_slices, figsize=(4 * num_slices, 8))        mid = D // 2        for i, si in enumerate(slice_indices):            axes[0, i].imshow(results["diffusion"][si], cmap="hot")            axes[0, i].set_title(f"D(x) slice {si}")            axes[0, i].axis("off")            axes[1, i].imshow(results["proliferation"][si], cmap="hot")            axes[1, i].set_title(f"ρ(x) slice {si}")            axes[1, i].axis("off")        plt.suptitle("Learned Physics Parameters", fontsize=14, fontweight="bold")        plt.tight_layout()        plt.savefig(os.path.join(output_dir, "physics_params.png"), dpi=150, bbox_inches="tight")        plt.close()    # --- 4. Growth Trajectory (if simulated) ---    if "growth_trajectory" in results:        traj = results["growth_trajectory"]        n_frames = min(8, len(traj))        frame_indices = np.linspace(0, len(traj) - 1, n_frames, dtype=int)        mid_slice = D // 2        fig, axes = plt.subplots(1, n_frames, figsize=(3 * n_frames, 3))        for i, fi in enumerate(frame_indices):            ax = axes[i] if n_frames > 1 else axes            ax.imshow(traj[fi][mid_slice], cmap=tumor_cmap, vmin=0, vmax=1)            ax.set_title(f"Day {fi}", fontsize=10)            ax.axis("off")        plt.suptitle("Tumor Growth Simulation", fontsize=14, fontweight="bold")        plt.tight_layout()        plt.savefig(os.path.join(output_dir, "growth_simulation.png"), dpi=150, bbox_inches="tight")        plt.close()    # --- 5. Volume Growth Curve ---    if "growth_trajectory" in results:        volumes = [(t > 0.5).sum() for t in results["growth_trajectory"]]        fig, ax = plt.subplots(figsize=(8, 5))        ax.plot(volumes, linewidth=2, color="#e74c3c")        ax.fill_between(range(len(volumes)), volumes, alpha=0.2, color="#e74c3c")        ax.set_xlabel("Time (days)", fontsize=12)        ax.set_ylabel("Tumor Volume (voxels)", fontsize=12)        ax.set_title("Predicted Tumor Volume Over Time", fontsize=14, fontweight="bold")        ax.grid(True, alpha=0.3)        plt.tight_layout()        plt.savefig(os.path.join(output_dir, "volume_curve.png"), dpi=150, bbox_inches="tight")        plt.close()    # --- 6. Uncertainty Map (if MC Dropout used) ---    if "uncertainty" in results:        unc = results["uncertainty"]        fig, axes = plt.subplots(1, num_slices, figsize=(4 * num_slices, 4))        for i, si in enumerate(slice_indices):            ax = axes[i] if num_slices > 1 else axes            im = ax.imshow(unc[si], cmap="magma")            ax.set_title(f"Slice {si}", fontsize=12)            ax.axis("off")                    plt.colorbar(im, ax=axes, shrink=0.8, label="Uncertainty (Std Dev)")        plt.suptitle("Tumor Density Uncertainty (MC Dropout)", fontsize=14, fontweight="bold")        plt.tight_layout()        plt.savefig(os.path.join(output_dir, "uncertainty.png"), dpi=150, bbox_inches="tight")        plt.close()    print(f"Visualizations saved to {output_dir}")def main():    parser = argparse.ArgumentParser(description="Brain Tumor Prediction")    parser.add_argument("--input_dir", type=str, required=True,                        help="Directory containing patient MRI files")    parser.add_argument("--checkpoint", type=str, required=True,                        help="Path to model checkpoint")    parser.add_argument("--output_dir", type=str, default="./predictions",                        help="Output directory for results")    parser.add_argument("--simulate_days", type=int, default=0,                        help="Days to simulate tumor growth (0 = no sim)")    parser.add_argument("--dt", type=float, default=1.0,                        help="Time step for simulation")    parser.add_argument("--mc_samples", type=int, default=0,                        help="Number of MC Dropout samples for uncertainty (0 to disable)")    parser.add_argument("--backbone", type=str, default="resnet50")    parser.add_argument("--device", type=str, default="cuda")    args = parser.parse_args()    config = get_config()    config.model.backbone = args.backbone    device = torch.device(args.device if torch.cuda.is_available() else "cpu")    # Load model    model = load_model(args.checkpoint, config, device)    # Load patient scan    print(f"Loading patient scan from {args.input_dir}...")    image, affine = load_patient_scan(args.input_dir, config.data.spatial_size)    # Predict    print("Running prediction...")    results = predict(model, image, device,                      simulate_days=args.simulate_days, dt=1.0)    # Print stats    print("\\n" + "=" * 50)    print("TUMOR ANALYSIS RESULTS")    print("=" * 50)    for k, v in results["stats"].items():        print(f"  {k}: {v}")    # Save outputs    os.makedirs(args.output_dir, exist_ok=True)    save_nifti(results["tumor_density"], affine,               os.path.join(args.output_dir, "tumor_density.nii.gz"))    if "segmentation" in results:        save_nifti(results["segmentation"].astype(np.float32), affine,                   os.path.join(args.output_dir, "segmentation.nii.gz"))    if results["diffusion"].ndim > 1:        save_nifti(results["diffusion"], affine,                   os.path.join(args.output_dir, "diffusion_D.nii.gz"))        save_nifti(results["proliferation"], affine,                   os.path.join(args.output_dir, "proliferation_rho.nii.gz"))    # Visualize    visualize_results(results, args.output_dir)    print(f"\\nAll outputs saved to {args.output_dir}")if __name__ == "__main__":    main()""")

In [ ]:
import rewith open("config.py", "r") as f: cfg = f.read()cfg = re.sub(r'num_epochs_pretrain: int = \d+', 'num_epochs_pretrain: int = 0', cfg)cfg = re.sub(r'num_epochs_finetune: int = \d+', 'num_epochs_finetune: int = 100', cfg)cfg = re.sub(r'batch_size: int = \d+', 'batch_size: int = 2', cfg)cfg = re.sub(r'num_workers: int = \d+', 'num_workers: int = 2', cfg)cfg = re.sub(r'device: str = "\w+"', 'device: str = "cuda"', cfg)with open("config.py", "w") as f: f.write(cfg)

## 📥 Download Dataset

In [ ]:
from monai.apps import download_and_extractimport globDATA_DIR = "./data"MSD_URL = "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task01_BrainTumour.tar"task_dir = os.path.join(DATA_DIR, "Task01_BrainTumour")if not os.path.exists(task_dir) or len(glob.glob(os.path.join(task_dir, "imagesTr", "*.nii.gz"))) == 0:    download_and_extract(url=MSD_URL, filepath=os.path.join(DATA_DIR, "Task01_BrainTumour.tar"), output_dir=DATA_DIR)

## ⚛️ Run Phase 2 (Physics Finetuning)

In [ ]:
# IMPORTANT: Replace the path below with your uploaded checkpoint path!CHECKPOINT_PATH = "/kaggle/input/YOUR_UPLOADED_PATH_HERE/pretrain_best.pth"!python train.py --data_root data/Task01_BrainTumour --phase finetune --device cuda --batch_size 1 --resume "{CHECKPOINT_PATH}"

## 🔮 Run Inference (Produce Final Images)

In [ ]:
import globsample = glob.glob("data/Task01_BrainTumour/imagesTr/BRATS_*.nii.gz")[0]!python predict.py --input_dir "{sample}" --checkpoint checkpoints/finetune_best.pth --simulate_days 30 --output_dir final_output --device cuda

In [ ]:
import shutil, globkaggle_out = "/kaggle/working/results"os.makedirs(kaggle_out, exist_ok=True)for f in glob.glob("checkpoints/*.pth") + glob.glob("final_output/*") + glob.glob("logs/*"):    shutil.copy2(f, kaggle_out)print("All saved to", kaggle_out)